In [ ]:
# Environment check. Prints everything mounted under /kaggle/input so we can
# confirm the EEG datasets attached before the pipeline runs.

import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub


In [ ]:
import os

# Walk the input tree until we hit a folder that actually holds EDF
# recordings plus a summary file. That folder is the CHB-MIT root.
for root, dirs, files in os.walk('/kaggle/input'):
    edf = [f for f in files if f.endswith('.edf')]
    summary = [f for f in files if 'summary' in f.lower() or f.endswith('.txt')]
    if edf or summary:
        print(root)
        print("  EDF files:", len(edf), "| example:", edf[:2])
        print("  summary/txt:", summary[:3])
        break  # drop this break to list every patient folder

In [ ]:
import os

BASE = '/kaggle/input/datasets/abhishekinnvonix/seizure-epilepcy-chb-mit-eeg-dataset-pediatric/chb-mit-scalp-eeg-database-1.0.0'

# Step 1 - raw data inventory. Patient folders are named chb01 .. chb24.
patient_dirs = sorted([d for d in os.listdir(BASE) if d.startswith('chb')])
print("Patient folders:", patient_dirs)
print("Count:", len(patient_dirs))

# Spot-check one patient so we know the folder layout
if patient_dirs:
    p0 = os.path.join(BASE, patient_dirs[0])
    files = sorted(os.listdir(p0))
    edf = [f for f in files if f.endswith('.edf')]
    summary = [f for f in files if 'summary' in f.lower()]
    print(f"\n{patient_dirs[0]}: {len(edf)} EDF files")
    print("  first EDFs:", edf[:3])
    print("  summary file:", summary)

# Grand total of recordings in the database
total_edf = sum(len([f for f in os.listdir(os.path.join(BASE, d)) if f.endswith('.edf')]) for d in patient_dirs)
print("\nTotal EDF files:", total_edf)

In [ ]:
# Utility: zip the figures folder so it can be pulled off the session at once.
import shutil
shutil.make_archive('/kaggle/working/all_figures', 'zip', '/kaggle/working/figures')

In [ ]:
import os
import re

BASE = '/kaggle/input/datasets/abhishekinnvonix/seizure-epilepcy-chb-mit-eeg-dataset-pediatric/chb-mit-scalp-eeg-database-1.0.0'

def parse_summary(patient):
    """Read chbXX-summary.txt and return one record per EDF file,
    each carrying that file's seizure start/end times."""
    summary_path = os.path.join(BASE, patient, f'{patient}-summary.txt')
    with open(summary_path, 'r') as f:
        text = f.read()

    records = []
    # The summary is one block per EDF file; split on the file-name marker
    blocks = re.split(r'File Name:', text)
    for block in blocks[1:]:
        fname_match = re.match(r'\s*(\S+\.edf)', block)
        if not fname_match:
            continue
        fname = fname_match.group(1)

        # Seizure count for this file (0 for the clean ones)
        nsz_match = re.search(r'Number of Seizures in File:\s*(\d+)', block)
        n_sz = int(nsz_match.group(1)) if nsz_match else 0

        # Pull every start/end pair - one file can carry more than one seizure
        starts = [int(x) for x in re.findall(r'Seizure(?:\s+\d+)?\s+Start Time:\s*(\d+)\s*seconds', block)]
        ends   = [int(x) for x in re.findall(r'Seizure(?:\s+\d+)?\s+End Time:\s*(\d+)\s*seconds', block)]
        seizures = list(zip(starts, ends))

        records.append({
            'patient': patient,
            'file': fname,
            'n_seizures': n_sz,
            'seizures': seizures,  # list of (start_sec, end_sec) within this file
        })
    return records

# Check the parser against chb01, where the seizure-bearing files are known
recs = parse_summary('chb01')
print(f"chb01: {len(recs)} files parsed")
for r in recs:
    if r['n_seizures'] > 0:
        print(f"  {r['file']}: {r['n_seizures']} seizure(s) -> {r['seizures']}")

# chb01 is documented with 7 seizures. Anything else means the regex is off.
total_sz = sum(r['n_seizures'] for r in recs)
print(f"\nchb01 total seizures: {total_sz} (expected 7)")

In [ ]:
import json

# Step 2 - run the parser over every patient and tally seizures.
inventory = {}
for patient in sorted([d for d in os.listdir(BASE) if d.startswith('chb')]):
    try:
        recs = parse_summary(patient)
    except FileNotFoundError:
        print(f"  WARNING: no summary for {patient}, skipping")
        continue
    total_sz = sum(r['n_seizures'] for r in recs)
    sz_files = [r['file'] for r in recs if r['n_seizures'] > 0]
    inventory[patient] = {
        'total_seizures': total_sz,
        'n_seizure_files': len(sz_files),
        'seizure_files': sz_files,
        'all_records': recs,
    }

# Per-patient table, flagging who clears the three-seizure bar
print(f"{'Patient':<10}{'Seizures':<10}{'SzFiles':<10}{'Eligible(>=3)'}")
print("-" * 45)
eligible = []
for p, info in inventory.items():
    elig = info['total_seizures'] >= 3
    if elig:
        eligible.append(p)
    print(f"{p:<10}{info['total_seizures']:<10}{info['n_seizure_files']:<10}{'YES' if elig else 'no'}")

print("-" * 45)
print(f"\nEligible patients (>=3 seizures): {len(eligible)}")
print(eligible)

# Cache to disk so later stages never re-parse the summary files
os.makedirs('/kaggle/working/artifacts', exist_ok=True)
with open('/kaggle/working/artifacts/inventory.json', 'w') as f:
    # all_records is bulky and only needed in-session, so save the slim version
    slim = {p: {k: v for k, v in info.items() if k != 'all_records'} for p, info in inventory.items()}
    slim['_eligible'] = eligible
    json.dump(slim, f, indent=2)
print("\nSaved: /kaggle/working/artifacts/inventory.json")

In [ ]:
import os, json, numpy as np, mne
mne.set_log_level('ERROR')  # otherwise MNE logs a line per file

BASE = '/kaggle/input/datasets/abhishekinnvonix/seizure-epilepcy-chb-mit-eeg-dataset-pediatric/chb-mit-scalp-eeg-database-1.0.0'

# ------------- Labelling parameters, fixed for the whole study -------------
SPH_MIN   = 5      # seizure prediction horizon (discarded gap before onset)
SOP_MIN   = 30     # seizure occurrence period (pre-ictal window)
INTERICTAL_GAP_H = 4   # interictal must be >=4h from any seizure on/offset
WIN_SEC   = 30     # window length, non-overlapping
FS        = 256    # CHB-MIT native sampling rate
BANDPASS  = (0.5, 50.0)
NOTCH     = 60.0   # US mains
# ---------------------------------------------------------------------------

def get_file_start_seconds(patient, records):
    """Kept as a stub on purpose.

    The summary files give wall-clock times, so stitching every recording into
    one continuous timeline means trusting those clocks across file gaps. We
    found that fragile. Instead each file is handled on its own: windows are
    labelled against the seizures inside that same file, and interictal data is
    only drawn from files that contain no seizure at all."""
    return None  # see label_file below for the per-file approach we use instead

def label_file(patient, rec, raw_dur_sec):
    """Label every 30 s window of one EDF file.

    Returns (win_start_sec, win_end_sec, label) triples, where label is one of
    'preictal', 'interictal' or 'discard', plus a flag for whether this file
    contained a seizure."""
    seizures = rec['seizures']  # [(start,end),...] in seconds within this file
    labels = []
    n_wins = int(raw_dur_sec // WIN_SEC)

    for w in range(n_wins):
        ws, we = w*WIN_SEC, (w+1)*WIN_SEC
        wc = (ws+we)/2  # window center
        lab = 'interictal_candidate'  # default; downgraded below

        if seizures:
            for (sz_s, sz_e) in seizures:
                preictal_start = sz_s - (SPH_MIN+SOP_MIN)*60
                preictal_end   = sz_s - SPH_MIN*60
                sph_start      = sz_s - SPH_MIN*60
                # the SOP window we want to forecast from
                if preictal_start <= wc < preictal_end:
                    lab = 'preictal'; break
                # the SPH gap just before onset: thrown away on purpose
                if sph_start <= wc < sz_s:
                    lab = 'discard'; break
                # seizure itself plus the post-ictal recovery buffer
                if sz_s <= wc < sz_e + INTERICTAL_GAP_H*3600:
                    lab = 'discard'; break
                # run-up to the preictal window: too close to call either way
                if preictal_start - INTERICTAL_GAP_H*3600 <= wc < preictal_start:
                    lab = 'discard'; break
        labels.append((ws, we, lab))
    return labels, (len(seizures) > 0)

# ---- Eyeball the labelling on one seizure file and one clean file ----
inv = json.load(open('/kaggle/working/artifacts/inventory.json'))
recs01 = parse_summary('chb01')

test_files = ['chb01_03.edf', 'chb01_01.edf']  # first has a seizure, second does not
for fname in test_files:
    rec = next(r for r in recs01 if r['file'] == fname)
    path = os.path.join(BASE, 'chb01', fname)
    raw = mne.io.read_raw_edf(path, preload=False, verbose='ERROR')
    dur = raw.n_times / raw.info['sfreq']
    labels, has_sz = label_file('chb01', rec, dur)
    from collections import Counter
    c = Counter(l[2] for l in labels)
    print(f"{fname}: dur={dur:.0f}s, seizures={rec['seizures']}, has_sz={has_sz}")
    print(f"  window labels: {dict(c)}")
    if 'preictal' in c:
        pre = [l for l in labels if l[2]=='preictal']
        print(f"  preictal spans: {pre[0][0]}s -> {pre[-1][1]}s (should end ~{rec['seizures'][0][0]-SPH_MIN*60}s)")
    print()

In [ ]:
import os, json, numpy as np, mne
from scipy import signal as sps
from scipy.stats import entropy as scipy_entropy
from collections import Counter
mne.set_log_level('ERROR')

BASE = '/kaggle/input/datasets/abhishekinnvonix/seizure-epilepcy-chb-mit-eeg-dataset-pediatric/chb-mit-scalp-eeg-database-1.0.0'
OUT = '/kaggle/working/artifacts'
os.makedirs(OUT, exist_ok=True)

SPH_MIN, SOP_MIN, INTERICTAL_GAP_H = 5, 30, 4
WIN_SEC, FS = 30, 256
BANDS = {'delta':(0.5,4),'theta':(4,8),'alpha':(8,13),'beta':(13,30),'gamma':(30,50)}
CLEAN_FILE_BUFFER = 2  # a clean file this close to a seizure file is still not trusted as interictal

def common_channels(patient, sz_files_any):
    """Channels present in every one of a patient's recordings.

    Montages drift between files, so we take the intersection and work with
    that fixed set throughout."""
    pdir = os.path.join(BASE, patient)
    edfs = sorted([f for f in os.listdir(pdir) if f.endswith('.edf')])
    ch_sets = []
    for f in edfs:
        try:
            raw = mne.io.read_raw_edf(os.path.join(pdir, f), preload=False)
            ch_sets.append(set(raw.ch_names))
        except Exception:
            continue
    common = set.intersection(*ch_sets) if ch_sets else set()
    # keep only bipolar EEG channels (contain '-'), drop dummy/status channels
    common = sorted([c for c in common if '-' in c and c.upper() not in ('.-.','-')])
    return common

def extract_features(win_data, fs=FS):
    """Flatten one (n_channels, n_samples) window into a feature vector."""
    feats = []
    for ch in win_data:
        # relative band power from a Welch periodogram
        f, pxx = sps.welch(ch, fs=fs, nperseg=min(256, len(ch)))
        total = np.trapz(pxx, f) + 1e-12
        for (lo, hi) in BANDS.values():
            idx = (f >= lo) & (f < hi)
            feats.append(np.trapz(pxx[idx], f[idx]) / total)
        # line length: total sample-to-sample movement
        feats.append(np.sum(np.abs(np.diff(ch))))
        # spectral entropy of the normalised spectrum
        p = pxx / (pxx.sum() + 1e-12)
        feats.append(scipy_entropy(p + 1e-12))
    return np.array(feats, dtype=np.float32)

def process_patient(patient):
    recs = parse_summary(patient)
    chans = common_channels(patient, None)
    if len(chans) < 16:
        return None  # too few channels, skip (report it)

    # summary order is chronological, so keep it
    file_order = [r['file'] for r in recs]
    sz_file_idx = set(i for i,r in enumerate(recs) if r['n_seizures'] > 0)
    # only clean files far enough from any seizure file qualify as interictal
    interictal_ok = set()
    for i, r in enumerate(recs):
        if r['n_seizures'] == 0:
            near = any(abs(i-j) <= CLEAN_FILE_BUFFER for j in sz_file_idx)
            if not near:
                interictal_ok.add(r['file'])

    X, y, seizure_id, seg_type = [], [], [], []
    sz_counter = 0
    for r in recs:
        fpath = os.path.join(BASE, patient, r['file'])
        try:
            raw = mne.io.read_raw_edf(fpath, preload=True)
        except Exception:
            continue
        raw.pick_channels(chans, ordered=True)
        raw.filter(0.5, 50., verbose='ERROR')
        try: raw.notch_filter(60., verbose='ERROR')
        except Exception: pass
        data = raw.get_data()
        dur = data.shape[1] / FS
        labels, has_sz = label_file(patient, r, dur)

        # tag each preictal window with its seizure, so splits can stay seizure-wise
        for (ws, we, lab) in labels:
            s0, s1 = int(ws*FS), int(we*FS)
            if s1 > data.shape[1]:
                continue
            if lab == 'preictal':
                win = data[:, s0:s1]
                X.append(extract_features(win)); y.append(1)
                # seizure id is a running counter over the patient's seizures
                seizure_id.append(f"{patient}_sz{sz_counter}")
                seg_type.append('preictal')
            elif lab == 'interictal_candidate' and r['file'] in interictal_ok:
                win = data[:, s0:s1]
                X.append(extract_features(win)); y.append(0)
                seizure_id.append(f"{patient}_inter")
                seg_type.append('interictal')
        if has_sz:
            sz_counter += r['n_seizures']

    if not X or sum(y) == 0:
        return None
    return {
        'patient': patient, 'channels': chans,
        'X': np.array(X), 'y': np.array(y),
        'seizure_id': np.array(seizure_id), 'seg_type': np.array(seg_type),
    }

# ---- Feature extraction across the whole database ----
eligible = json.load(open(f'{OUT}/inventory.json'))['_eligible']
report = {}
for p in eligible:
    print(f"Processing {p}...", flush=True)
    res = process_patient(p)
    if res is None:
        report[p] = 'SKIPPED (channels/seizures)'
        print(f"  {p}: skipped"); continue
    np.savez_compressed(f'{OUT}/{p}_features.npz',
                        X=res['X'], y=res['y'],
                        seizure_id=res['seizure_id'], seg_type=res['seg_type'])
    n_pre, n_int = int((res['y']==1).sum()), int((res['y']==0).sum())
    report[p] = {'n_channels': len(res['channels']),
                 'n_preictal': n_pre, 'n_interictal': n_int,
                 'n_unique_seizures': len(set(res['seizure_id'])) - (1 if n_int>0 else 0)}
    print(f"  {p}: {len(res['channels'])}ch, pre={n_pre}, inter={n_int}")

json.dump(report, open(f'{OUT}/feature_report.json','w'), indent=2)
print("\nDONE. Report saved to feature_report.json")
print(json.dumps(report, indent=2))

In [ ]:
import json, numpy as np, os
OUT = '/kaggle/working/artifacts'
report = json.load(open(f'{OUT}/feature_report.json'))

# Step 3 - final cohort. A patient is usable only if both classes survived
# the labelling rules above.
cohort = [p for p,v in report.items()
          if isinstance(v, dict) and v['n_interictal'] > 0 and v['n_preictal'] > 0]
excluded = [p for p in report if p not in cohort]

print(f"FINAL COHORT: N={len(cohort)}")
print(cohort)
print(f"\nEXCLUDED (N={len(excluded)}): {excluded}")

# How lopsided each patient's class balance ended up
print(f"\n{'Patient':<9}{'Pre':<7}{'Inter':<8}{'Ratio':<10}{'Seizures':<9}{'Ch'}")
print("-"*50)
for p in cohort:
    v = report[p]
    ratio = v['n_interictal'] / max(v['n_preictal'],1)
    print(f"{p:<9}{v['n_preictal']:<7}{v['n_interictal']:<8}1:{ratio:<8.1f}{v['n_unique_seizures']:<9}{v['n_channels']}")

json.dump({'cohort':cohort,'excluded':excluded}, open(f'{OUT}/cohort.json','w'), indent=2)
print("\nSaved cohort.json")

In [ ]:
import json, numpy as np, os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
np.random.seed(42)

OUT = '/kaggle/working/artifacts'
cohort = json.load(open(f'{OUT}/cohort.json'))['cohort']

def load_patient(p):
    d = np.load(f'{OUT}/{p}_features.npz', allow_pickle=True)
    return d['X'], d['y'], d['seizure_id'], d['seg_type']

def conformal_fnr_threshold(scores_pre_cal, alpha):
    """Split-conformal threshold aimed at holding the miss rate at or below alpha.

    scores_pre_cal holds P(preictal) for the calibration preictal windows. We
    raise an alarm when a score reaches tau, so a preictal window is missed
    whenever its score falls short. Putting tau at the alpha-quantile of the
    calibration scores (with the finite-sample correction) is what bounds that
    miss rate."""
    n = len(scores_pre_cal)
    if n == 0:
        return 0.0
    # finite-sample corrected quantile level
    q_level = np.floor(alpha * (n + 1)) / n
    q_level = min(max(q_level, 0.0), 1.0)
    tau = np.quantile(scores_pre_cal, q_level, method='lower')
    return tau

def eval_fnr_fpr(model, scaler, X_pre, X_int, tau):
    fnr = fpr = np.nan
    if len(X_pre):
        s = model.predict_proba(scaler.transform(X_pre))[:,1]
        fnr = np.mean(s < tau)         # missed preictal
    if len(X_int):
        s = model.predict_proba(scaler.transform(X_int))[:,1]
        fpr = np.mean(s >= tau)        # false alarms on interictal
    return fnr, fpr

ALPHA = 0.10  # target: miss at most 10% of preictal windows
results = {'random':{}, 'time':{}}

for p in cohort:
    X, y, sid, seg = load_patient(p)
    pre_mask, int_mask = (y==1), (y==0)
    Xp, Xi = X[pre_mask], X[int_mask]
    sid_p = sid[pre_mask]
    uniq_sz = sorted(set(sid_p.tolist()))
    if len(uniq_sz) < 2:
        continue

    for split_type in ['random','time']:
        if split_type == 'time':
            # chronological split: earliest half of the seizures calibrate, rest are test
            k = len(uniq_sz); n_cal = int(np.ceil(k/2))
            cal_sz, test_sz = set(uniq_sz[:n_cal]), set(uniq_sz[n_cal:])
        else:
            # same split size, but seizures drawn at random
            perm = list(np.random.permutation(uniq_sz))
            n_cal = int(np.ceil(len(perm)/2))
            cal_sz, test_sz = set(perm[:n_cal]), set(perm[n_cal:])

        cal_pre = Xp[np.isin(sid_p, list(cal_sz))]
        test_pre = Xp[np.isin(sid_p, list(test_sz))]
        # interictal split the same way: first/second half for time, shuffled for random
        ni = len(Xi)
        if split_type=='time':
            cut = ni//2; cal_int, test_int = Xi[:cut], Xi[cut:]
        else:
            idx = np.random.permutation(ni); cut=ni//2
            cal_int, test_int = Xi[idx[:cut]], Xi[idx[cut:]]

        if len(cal_pre)<5 or len(test_pre)<1:
            continue

        # ---- backbone classifier, trained only on the calibration seizures ----
        n_bal = min(len(cal_pre)*3, len(cal_int))
        cal_int_bal = cal_int[np.random.choice(len(cal_int), n_bal, replace=False)]
        Xtr = np.vstack([cal_pre, cal_int_bal])
        ytr = np.r_[np.ones(len(cal_pre)), np.zeros(len(cal_int_bal))]
        scaler = StandardScaler().fit(Xtr)
        model = LogisticRegression(max_iter=1000, class_weight='balanced').fit(scaler.transform(Xtr), ytr)

        # ---- the obvious conformal threshold, taken from calibration preictal scores ----
        s_cal_pre = model.predict_proba(scaler.transform(cal_pre))[:,1]
        tau = conformal_fnr_threshold(s_cal_pre, ALPHA)

        # ---- evaluate on seizures the model has never seen ----
        fnr, fpr = eval_fnr_fpr(model, scaler, test_pre, test_int, tau)
        results[split_type][p] = {'realized_fnr':float(fnr), 'realized_fpr':float(fpr),
                                  'target_fnr':ALPHA, 'n_cal_pre':int(len(cal_pre)),
                                  'n_test_pre':int(len(test_pre))}

json.dump(results, open(f'{OUT}/stage5_naive.json','w'), indent=2)

# ---- does the guarantee survive the transfer? ----
for st in ['random','time']:
    r = results[st]
    fnrs = [v['realized_fnr'] for v in r.values() if not np.isnan(v['realized_fnr'])]
    violations = sum(f > ALPHA + 0.02 for f in fnrs)  # small tolerance
    print(f"\n[{st.upper()} split]  target FNR={ALPHA}")
    print(f"  patients: {len(fnrs)}, mean realized FNR: {np.mean(fnrs):.3f}")
    print(f"  GUARANTEE VIOLATIONS (FNR>target): {violations}/{len(fnrs)}")
    print(f"  worst-patient FNR: {np.max(fnrs):.3f}")

In [ ]:
import numpy as np, json
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
OUT='/kaggle/working/artifacts'
cohort=json.load(open(f'{OUT}/cohort.json'))['cohort']

def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id'],d['seg_type']

for p in ['chb15','chb04']:  # the most and least balanced patients
    X,y,sid,seg=load_patient(p)
    Xp,Xi=X[y==1],X[y==0]
    print(f"\n=== {p} ===")
    print(f"  X shape {X.shape}, preictal {len(Xp)}, interictal {len(Xi)}")
    print(f"  X any NaN? {np.isnan(X).any()}  any Inf? {np.isinf(X).any()}")
    print(f"  X range: min {X.min():.3g}, max {X.max():.3g}, mean {X.mean():.3g}")

    # quick window-level split, purely to confirm the features separate at all
    n=min(len(Xp),len(Xi))
    idxp=np.random.permutation(len(Xp))[:n]; idxi=np.random.permutation(len(Xi))[:n]
    Xall=np.vstack([Xp[idxp],Xi[idxi]]); yall=np.r_[np.ones(n),np.zeros(n)]
    perm=np.random.permutation(2*n); Xall,yall=Xall[perm],yall[perm]
    cut=int(0.6*len(Xall))
    sc=StandardScaler().fit(Xall[:cut])
    m=LogisticRegression(max_iter=1000,class_weight='balanced').fit(sc.transform(Xall[:cut]),yall[:cut])
    s=m.predict_proba(sc.transform(Xall[cut:]))[:,1]
    yt=yall[cut:]
    print(f"  train acc {m.score(sc.transform(Xall[:cut]),yall[:cut]):.3f}")
    print(f"  test preictal-score mean {s[yt==1].mean():.3f} vs interictal-score mean {s[yt==0].mean():.3f}")
    print(f"  --> if these two means are close, features aren't separating classes")

In [ ]:
import json, numpy as np, os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
OUT='/kaggle/working/artifacts'
cohort=json.load(open(f'{OUT}/cohort.json'))['cohort']

def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id'],d['seg_type']

def conformal_fnr_threshold(scores_pre_cal, alpha):
    """Same threshold as above, written against the order statistics directly.

    Preictal windows score high and we alarm at tau, so a miss is a score
    below tau. Taking the floor(alpha*(n+1))-th smallest calibration score is
    the conformal way to bound that miss probability by alpha."""
    s = np.sort(np.asarray(scores_pre_cal))   # ascending
    n = len(s)
    if n == 0: return 0.0
    # largest tau that still keeps at most an alpha fraction underneath it
    k = int(np.floor(alpha * (n + 1)))         # conformal rank
    k = min(max(k, 0), n)                       # clamp
    if k == 0:
        return s[0] - 1e-9                       # threshold below all -> FNR ~0
    return s[k-1]                               # k-th smallest score

def eval_fnr_fpr(model, scaler, X_pre, X_int, tau):
    fnr=fpr=np.nan
    if len(X_pre):
        s=model.predict_proba(scaler.transform(X_pre))[:,1]; fnr=np.mean(s<tau)
    if len(X_int):
        s=model.predict_proba(scaler.transform(X_int))[:,1]; fpr=np.mean(s>=tau)
    return fnr,fpr

ALPHA=0.10
results={'random':{},'time':{}}
for p in cohort:
    X,y,sid,seg=load_patient(p)
    Xp,Xi=X[y==1],X[y==0]; sid_p=sid[y==1]
    uniq=sorted(set(sid_p.tolist()))
    if len(uniq)<2: continue
    for st in ['random','time']:
        if st=='time':
            k=len(uniq); nc=int(np.ceil(k/2))
            cal_sz,test_sz=set(uniq[:nc]),set(uniq[nc:])
        else:
            perm=list(np.random.permutation(uniq)); nc=int(np.ceil(len(perm)/2))
            cal_sz,test_sz=set(perm[:nc]),set(perm[nc:])
        cal_pre=Xp[np.isin(sid_p,list(cal_sz))]; test_pre=Xp[np.isin(sid_p,list(test_sz))]
        ni=len(Xi)
        if st=='time': cut=ni//2; cal_int,test_int=Xi[:cut],Xi[cut:]
        else:
            idx=np.random.permutation(ni); cut=ni//2
            cal_int,test_int=Xi[idx[:cut]],Xi[idx[cut:]]
        if len(cal_pre)<5 or len(test_pre)<1: continue
        n_bal=min(len(cal_pre)*3,len(cal_int))
        cbi=cal_int[np.random.choice(len(cal_int),n_bal,replace=False)]
        Xtr=np.vstack([cal_pre,cbi]); ytr=np.r_[np.ones(len(cal_pre)),np.zeros(len(cbi))]
        sc=StandardScaler().fit(Xtr)
        m=LogisticRegression(max_iter=1000,class_weight='balanced').fit(sc.transform(Xtr),ytr)
        s_cal=m.predict_proba(sc.transform(cal_pre))[:,1]
        tau=conformal_fnr_threshold(s_cal,ALPHA)
        fnr,fpr=eval_fnr_fpr(m,sc,test_pre,test_int,tau)
        results[st][p]={'realized_fnr':float(fnr),'realized_fpr':float(fpr),
                        'target':ALPHA,'n_cal_pre':int(len(cal_pre)),'tau':float(tau)}
json.dump(results,open(f'{OUT}/stage5_naive.json','w'),indent=2)

for st in ['random','time']:
    r=results[st]; fnrs=[v['realized_fnr'] for v in r.values() if not np.isnan(v['realized_fnr'])]
    fprs=[v['realized_fpr'] for v in r.values() if not np.isnan(v['realized_fpr'])]
    viol=sum(f>ALPHA+0.02 for f in fnrs)
    print(f"\n[{st.upper()}] target FNR={ALPHA}")
    print(f"  patients {len(fnrs)}, mean FNR {np.mean(fnrs):.3f}, worst {np.max(fnrs):.3f}")
    print(f"  violations {viol}/{len(fnrs)}, mean FPR (alarm burden) {np.mean(fprs):.3f}")

In [ ]:
import numpy as np, json
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
OUT='/kaggle/working/artifacts'
def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id'],d['seg_type']

p='chb15'
X,y,sid,seg=load_patient(p)
Xp,Xi=X[y==1],X[y==0]; sid_p=sid[y==1]
uniq=sorted(set(sid_p.tolist()))
perm=list(np.random.permutation(uniq)); nc=int(np.ceil(len(perm)/2))
cal_sz,test_sz=set(perm[:nc]),set(perm[nc:])
cal_pre=Xp[np.isin(sid_p,list(cal_sz))]; test_pre=Xp[np.isin(sid_p,list(test_sz))]
ni=len(Xi); idx=np.random.permutation(ni); cut=ni//2
cal_int,test_int=Xi[idx[:cut]],Xi[idx[cut:]]

n_bal=min(len(cal_pre)*3,len(cal_int))
cbi=cal_int[np.random.choice(len(cal_int),n_bal,replace=False)]
Xtr=np.vstack([cal_pre,cbi]); ytr=np.r_[np.ones(len(cal_pre)),np.zeros(len(cbi))]
sc=StandardScaler().fit(Xtr)
m=LogisticRegression(max_iter=1000,class_weight='balanced').fit(sc.transform(Xtr),ytr)

s_cal=m.predict_proba(sc.transform(cal_pre))[:,1]
s_test_pre=m.predict_proba(sc.transform(test_pre))[:,1]
s_test_int=m.predict_proba(sc.transform(test_int))[:,1]

alpha=0.10
s=np.sort(s_cal); n=len(s)
k=int(np.floor(alpha*(n+1))); k=min(max(k,0),n)
tau=s[k-1] if k>0 else s[0]-1e-9

print(f"cal preictal scores: n={n}")
print(f"  min {s.min():.3f}, 10th pct {np.percentile(s,10):.3f}, median {np.median(s):.3f}, max {s.max():.3f}")
print(f"conformal k={k}, tau={tau:.4f}")
print(f"test preictal scores: mean {s_test_pre.mean():.3f}, frac < tau (=FNR) {np.mean(s_test_pre<tau):.3f}")
print(f"test interict scores: mean {s_test_int.mean():.3f}, frac >= tau (=FPR) {np.mean(s_test_int>=tau):.3f}")
print(f"\nsample cal preictal scores: {np.round(s[:10],3)}")
print(f"sample test preictal scores: {np.round(s_test_pre[:10],3)}")

In [ ]:
import json, numpy as np, os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
OUT='/kaggle/working/artifacts'
cohort=json.load(open(f'{OUT}/cohort.json'))['cohort']

def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id'],d['seg_type']

ALPHA=0.10

def static_conformal_tau(scores_pre, alpha):
    s=np.sort(scores_pre); n=len(s)
    if n==0: return 0.0
    k=int(np.floor(alpha*(n+1))); k=min(max(k,0),n)
    return (s[0]-1e-9) if k==0 else s[k-1]

def train_backbone(cal_pre, cal_int):
    n_bal=min(len(cal_pre)*3,len(cal_int))
    cbi=cal_int[np.random.choice(len(cal_int),n_bal,replace=False)]
    Xtr=np.vstack([cal_pre,cbi]); ytr=np.r_[np.ones(len(cal_pre)),np.zeros(len(cbi))]
    sc=StandardScaler().fit(Xtr)
    m=LogisticRegression(max_iter=1000,class_weight='balanced').fit(sc.transform(Xtr),ytr)
    return m,sc

def run_patient(p, adaptive, gamma=0.05):
    """Walk a patient's held-out seizures in the order they happened.

    static  - tau is fixed once from the calibration seizures.
    adaptive - tau moves after each seizure in response to the error so far,
               the usual online-conformal update."""
    X,y,sid,seg=load_patient(p)
    Xp,Xi=X[y==1],X[y==0]; sid_p=sid[y==1]
    uniq=sorted(set(sid_p.tolist()))
    if len(uniq)<3: return None            # need >=1 cal + >=2 test seizures for adaptation
    k=len(uniq); nc=max(1,int(np.ceil(k/2)))
    cal_sz, test_sz = uniq[:nc], uniq[nc:]  # TIME-ordered
    cal_pre=Xp[np.isin(sid_p,cal_sz)]
    if len(cal_pre)<5: return None
    ni=len(Xi); cut=ni//2; cal_int,test_int_all=Xi[:cut],Xi[cut:]
    m,sc=train_backbone(cal_pre,cal_int)

    s_cal=m.predict_proba(sc.transform(cal_pre))[:,1]
    tau=static_conformal_tau(s_cal,ALPHA)   # both start from conformal tau

    # interictal scores, split across the test seizures for false-alarm accounting
    s_int_test=m.predict_proba(sc.transform(test_int_all))[:,1]
    int_chunks=np.array_split(s_int_test, len(test_sz))

    fnr_seq, fpr_seq, tau_seq = [], [], []
    for j, szid in enumerate(test_sz):
        tp=Xp[sid_p==szid]
        s=m.predict_proba(sc.transform(tp))[:,1]
        fnr = np.mean(s<tau) if len(s) else np.nan
        fpr = np.mean(int_chunks[j]>=tau) if len(int_chunks[j]) else np.nan
        fnr_seq.append(fnr); fpr_seq.append(fpr); tau_seq.append(tau)
        if adaptive and not np.isnan(fnr):
            # nudge tau up when we are missing less than budgeted, down when missing more
            tau = tau + gamma*(fnr - ALPHA)     # fnr>alpha -> tau up? NO: see note
            tau = float(np.clip(tau,0.0,1.0))
    return {'fnr':fnr_seq,'fpr':fpr_seq,'tau':tau_seq,
            'mean_fnr':float(np.nanmean(fnr_seq)),'mean_fpr':float(np.nanmean(fpr_seq)),
            'n_test_sz':len(test_sz)}

In [ ]:
import json, numpy as np, os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
OUT='/kaggle/working/artifacts'
cohort=json.load(open(f'{OUT}/cohort.json'))['cohort']

def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id'],d['seg_type']

ALPHA=0.10

def static_conformal_tau(scores_pre, alpha):
    s=np.sort(scores_pre); n=len(s)
    if n==0: return 0.0
    k=int(np.floor(alpha*(n+1))); k=min(max(k,0),n)
    return (s[0]-1e-9) if k==0 else s[k-1]

def train_backbone(cal_pre, cal_int):
    n_bal=min(len(cal_pre)*3,len(cal_int))
    cbi=cal_int[np.random.choice(len(cal_int),n_bal,replace=False)]
    Xtr=np.vstack([cal_pre,cbi]); ytr=np.r_[np.ones(len(cal_pre)),np.zeros(len(cbi))]
    sc=StandardScaler().fit(Xtr)
    m=LogisticRegression(max_iter=1000,class_weight='balanced').fit(sc.transform(Xtr),ytr)
    return m,sc

def run_patient(p, adaptive, gamma=0.05):
    X,y,sid,seg=load_patient(p)
    Xp,Xi=X[y==1],X[y==0]; sid_p=sid[y==1]
    uniq=sorted(set(sid_p.tolist()))
    if len(uniq)<3: return None
    k=len(uniq); nc=max(1,int(np.ceil(k/2)))
    cal_sz, test_sz = uniq[:nc], uniq[nc:]
    cal_pre=Xp[np.isin(sid_p,cal_sz)]
    if len(cal_pre)<5: return None
    ni=len(Xi); cut=ni//2; cal_int,test_int_all=Xi[:cut],Xi[cut:]
    m,sc=train_backbone(cal_pre,cal_int)

    s_cal=m.predict_proba(sc.transform(cal_pre))[:,1]
    tau=static_conformal_tau(s_cal,ALPHA)

    s_int_test=m.predict_proba(sc.transform(test_int_all))[:,1]
    int_chunks=np.array_split(s_int_test, len(test_sz))

    fnr_seq, fpr_seq, tau_seq = [], [], []
    for j, szid in enumerate(test_sz):
        tp=Xp[sid_p==szid]
        s=m.predict_proba(sc.transform(tp))[:,1]
        fnr = np.mean(s<tau) if len(s) else np.nan
        fpr = np.mean(int_chunks[j]>=tau) if len(int_chunks[j]) else np.nan
        fnr_seq.append(fnr); fpr_seq.append(fpr); tau_seq.append(float(tau))
        if adaptive and not np.isnan(fnr):
            tau = tau - gamma*(fnr - ALPHA)      # fnr>alpha -> tau DOWN -> catch more preictal
            tau = float(np.clip(tau,0.0,1.0))
    return {'fnr':fnr_seq,'fpr':fpr_seq,'tau':tau_seq,
            'mean_fnr':float(np.nanmean(fnr_seq)),'mean_fpr':float(np.nanmean(fpr_seq)),
            'n_test_sz':len(test_sz)}

static_res, adapt_res = {}, {}
for p in cohort:
    rs=run_patient(p, adaptive=False)
    ra=run_patient(p, adaptive=True, gamma=0.05)
    if rs: static_res[p]=rs
    if ra: adapt_res[p]=ra

def summarize(res, name):
    mf=[v['mean_fnr'] for v in res.values()]
    mp=[v['mean_fpr'] for v in res.values()]
    viol=sum(f>ALPHA+0.02 for f in mf)
    print(f"\n[{name}] target FNR={ALPHA}, patients={len(mf)}")
    print(f"  mean FNR {np.nanmean(mf):.3f}, worst {np.nanmax(mf):.3f}, violations {viol}/{len(mf)}")
    print(f"  mean FPR (alarm burden) {np.nanmean(mp):.3f}")

summarize(static_res,"STATIC conformal (time-split)")
summarize(adapt_res,"ADAPTIVE conformal (time-split)")
json.dump({'static':static_res,'adaptive':adapt_res}, open(f'{OUT}/stage5_adaptive.json','w'), indent=2)
print("\nSaved stage5_adaptive.json")

In [ ]:
import json, numpy as np, os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
OUT='/kaggle/working/artifacts'
cohort=json.load(open(f'{OUT}/cohort.json'))['cohort']

def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id'],d['seg_type']

TARGET_SENS=0.90   # catch at least 9 seizures in 10

def train_backbone(cal_pre,cal_int):
    n_bal=min(len(cal_pre)*3,len(cal_int))
    cbi=cal_int[np.random.choice(len(cal_int),n_bal,replace=False)]
    Xtr=np.vstack([cal_pre,cbi]); ytr=np.r_[np.ones(len(cal_pre)),np.zeros(len(cbi))]
    sc=StandardScaler().fit(Xtr)
    m=LogisticRegression(max_iter=1000,class_weight='balanced',C=0.5).fit(sc.transform(Xtr),ytr)
    return m,sc

def seizure_maxscore(m,sc,Xp,sid_p,sz_ids):
    """One score per seizure: the highest-scoring window in its preictal run."""
    out={}
    for szid in sz_ids:
        tp=Xp[sid_p==szid]
        if len(tp)==0: continue
        s=m.predict_proba(sc.transform(tp))[:,1]
        out[szid]=float(s.max())   # seizure caught if max >= tau
    return out

def conformal_sens_tau(cal_maxscores, target_sens):
    """Threshold on seizure-level scores rather than window-level ones.

    A seizure counts as missed when its best window still falls below tau, so
    tau sits at the alpha-quantile of the calibration seizures' peak scores."""
    alpha=1-target_sens
    s=np.sort(list(cal_maxscores.values())); n=len(s)
    if n==0: return 0.0
    k=int(np.floor(alpha*(n+1))); k=min(max(k,0),n)
    return (s[0]-1e-9) if k==0 else s[k-1]

def run_patient(p, adaptive, gamma=0.05):
    X,y,sid,seg=load_patient(p)
    Xp,Xi=X[y==1],X[y==0]; sid_p=sid[y==1]
    uniq=sorted(set(sid_p.tolist()))
    if len(uniq)<3: return None
    k=len(uniq); nc=max(2,int(np.ceil(k/2)))   # >=2 cal seizures so tau is defined
    if nc>=k: nc=k-1
    cal_sz,test_sz=uniq[:nc],uniq[nc:]
    if len(test_sz)<1: return None
    cal_pre=Xp[np.isin(sid_p,cal_sz)]
    ni=len(Xi); cut=ni//2; cal_int,test_int=Xi[:cut],Xi[cut:]
    m,sc=train_backbone(cal_pre,cal_int)

    cal_max=seizure_maxscore(m,sc,Xp,sid_p,cal_sz)
    tau=conformal_sens_tau(cal_max,TARGET_SENS)

    # false-alarm rate: share of interictal windows that clear tau (stands in for alarms/hour)
    s_int=m.predict_proba(sc.transform(test_int))[:,1]
    fa_rate=float(np.mean(s_int>=tau))

    caught=[]; tau_seq=[]
    for szid in test_sz:
        tp=Xp[sid_p==szid]
        smax=m.predict_proba(sc.transform(tp))[:,1].max() if len(tp) else 0.0
        hit=int(smax>=tau); caught.append(hit); tau_seq.append(float(tau))
        if adaptive:
            miss=1-hit
            tau=float(np.clip(tau - gamma*(miss-(1-TARGET_SENS)),0.0,1.0))
    sens=float(np.mean(caught))
    return {'sens':sens,'fa_rate':fa_rate,'n_test_sz':len(test_sz),'caught':caught}

for mode,adapt in [('STATIC',False),('ADAPTIVE',True)]:
    res={}
    for p in cohort:
        r=run_patient(p,adaptive=adapt)
        if r: res[p]=r
    sens=[v['sens'] for v in res.values()]
    fa=[v['fa_rate'] for v in res.values()]
    meets=sum(s>=TARGET_SENS-0.001 for s in sens)
    print(f"\n[{mode}] target sensitivity>={TARGET_SENS}, patients={len(sens)}")
    print(f"  mean sensitivity {np.mean(sens):.3f}, worst {np.min(sens):.3f}")
    print(f"  patients meeting guarantee: {meets}/{len(sens)}")
    print(f"  mean interictal false-alarm rate: {np.mean(fa):.3f}")
    json.dump(res,open(f'{OUT}/stage5_seizurelevel_{mode.lower()}.json','w'),indent=2)

In [ ]:
import json, numpy as np, os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.covariance import EmpiricalCovariance
np.random.seed(42)
OUT='/kaggle/working/artifacts'
cohort=json.load(open(f'{OUT}/cohort.json'))['cohort']
TARGET_SENS=0.90

def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id'],d['seg_type']

def seizure_maxscores(score_fn, Xp, sid_p, sz_ids):
    return {sz: float(score_fn(Xp[sid_p==sz]).max()) for sz in sz_ids if (sid_p==sz).sum()>0}

def conformal_tau(cal_max_vals, target_sens):
    alpha=1-target_sens; s=np.sort(cal_max_vals); n=len(s)
    if n==0: return 0.0
    k=int(np.floor(alpha*(n+1))); k=min(max(k,0),n)
    return (s[0]-1e-9) if k==0 else s[k-1]

def run_method(method):
    per_patient=[]
    fa_all=[]
    for p in cohort:
        X,y,sid,seg=load_patient(p)
        Xp,Xi=X[y==1],X[y==0]; sid_p=sid[y==1]
        uniq=sorted(set(sid_p.tolist()))
        if len(uniq)<3: continue

        # leave-one-seizure-out: every seizure takes a turn as the held-out one
        caught=[]
        for held in uniq:
            cal_sz=[s for s in uniq if s!=held]
            cal_pre=Xp[np.isin(sid_p,cal_sz)]
            if len(cal_pre)<5: continue

            if method=='logreg_pca':
                nb=min(len(cal_pre)*3,len(Xi))
                cbi=Xi[np.random.choice(len(Xi),nb,replace=False)]
                Xtr=np.vstack([cal_pre,cbi]); ytr=np.r_[np.ones(len(cal_pre)),np.zeros(len(cbi))]
                sc=StandardScaler().fit(Xtr)
                pca=PCA(n_components=min(20,Xtr.shape[1])).fit(sc.transform(Xtr))
                m=LogisticRegression(max_iter=1000,class_weight='balanced',C=0.1).fit(pca.transform(sc.transform(Xtr)),ytr)
                score_fn=lambda A: m.predict_proba(pca.transform(sc.transform(A)))[:,1]
            elif method=='mahalanobis':
                # the anomaly model never sees a preictal window - it is fit on interictal only
                sc=StandardScaler().fit(Xi)
                cov=EmpiricalCovariance().fit(sc.transform(Xi))
                score_fn=lambda A: cov.mahalanobis(sc.transform(A))  # higher=more anomalous

            # tau from the remaining seizures' peak scores
            cal_max=seizure_maxscores(score_fn,Xp,sid_p,cal_sz)
            tau=conformal_tau(list(cal_max.values()),TARGET_SENS)
            # did the held-out seizure clear it?
            tp=Xp[sid_p==held]
            hit=int(score_fn(tp).max()>=tau) if len(tp) else 0
            caught.append(hit)
            # false-alarm rate on an interictal sample
            si=score_fn(Xi[np.random.choice(len(Xi),min(500,len(Xi)),replace=False)])
            fa_all.append(np.mean(si>=tau))
        if caught:
            per_patient.append(np.mean(caught))
    return per_patient, fa_all

for method in ['logreg_pca','mahalanobis']:
    sens,fa=run_method(method)
    meets=sum(s>=TARGET_SENS-0.001 for s in sens)
    print(f"\n[{method} | LOSO] target sens>={TARGET_SENS}, patients={len(sens)}")
    print(f"  mean sensitivity {np.mean(sens):.3f}, worst {np.min(sens):.3f}")
    print(f"  meeting guarantee: {meets}/{len(sens)}, mean FA rate {np.mean(fa):.3f}")

In [ ]:
import json, numpy as np, os
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EmpiricalCovariance
np.random.seed(42)
OUT='/kaggle/working/artifacts'
cohort=json.load(open(f'{OUT}/cohort.json'))['cohort']

def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id'],d['seg_type']

def seizure_agg(scores, agg='max', topk=3):
    if len(scores)==0: return -np.inf
    if agg=='max': return float(np.max(scores))
    if agg=='topk': return float(np.mean(np.sort(scores)[-topk:]))
    if agg=='persist':  # max over 3-consecutive-window means
        if len(scores)<3: return float(np.max(scores))
        return float(np.max([scores[i:i+3].mean() for i in range(len(scores)-2)]))

def conformal_tau(vals, target_sens):
    alpha=1-target_sens; s=np.sort(vals); n=len(s)
    if n==0: return -np.inf
    k=int(np.floor(alpha*(n+1))); k=min(max(k,0),n)
    return (s[0]-1e-9) if k==0 else s[k-1]

def run(agg, target_cal):
    """Sweep the seizure-level aggregation and the calibration target.

    target_cal is what we calibrate against, which can sit above 0.9 to buy
    some headroom over the sensitivity we actually want."""
    sens_list, fa_list = [], []
    for p in cohort:
        X,y,sid,seg=load_patient(p)
        Xp,Xi=X[y==1],X[y==0]; sid_p=sid[y==1]
        uniq=sorted(set(sid_p.tolist()))
        if len(uniq)<3: continue
        caught=[]; fap=[]
        for held in uniq:
            cal_sz=[s for s in uniq if s!=held]
            sc=StandardScaler().fit(Xi)
            cov=EmpiricalCovariance().fit(sc.transform(Xi))
            score_fn=lambda A: cov.mahalanobis(sc.transform(A))
            cal_vals=[seizure_agg(score_fn(Xp[sid_p==s]),agg) for s in cal_sz]
            tau=conformal_tau(cal_vals,target_cal)
            tp=Xp[sid_p==held]
            caught.append(int(seizure_agg(score_fn(tp),agg)>=tau))
            si=score_fn(Xi[np.random.choice(len(Xi),min(500,len(Xi)),replace=False)])
            fap.append(np.mean(si>=tau))
        if caught:
            sens_list.append(np.mean(caught)); fa_list.append(np.mean(fap))
    return sens_list, fa_list

for agg in ['max','topk','persist']:
    for tcal in [0.90, 0.95, 0.99]:
        sens,fa=run(agg,tcal)
        meets=sum(s>=0.90-0.001 for s in sens)
        print(f"[agg={agg:7s} cal={tcal}] mean_sens {np.mean(sens):.3f}  worst {np.min(sens):.3f}  meets>=0.9 {meets}/{len(sens)}  FA {np.mean(fa):.3f}")

In [ ]:
import json, numpy as np, os
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EmpiricalCovariance
np.random.seed(42)
OUT='/kaggle/working/artifacts'; os.makedirs(f'{OUT}/figures',exist_ok=True)
cohort=json.load(open(f'{OUT}/cohort.json'))['cohort']

def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id'],d['seg_type']

# Step 4 - per-seizure separability. For each seizure, how far does its
# preictal anomaly score sit above that patient's own interictal spread?
records=[]
for p in cohort:
    X,y,sid,seg=load_patient(p)
    Xp,Xi=X[y==1],X[y==0]; sid_p=sid[y==1]
    uniq=sorted(set(sid_p.tolist()))
    if len(uniq)<3: continue
    sc=StandardScaler().fit(Xi)
    cov=EmpiricalCovariance().fit(sc.transform(Xi))
    int_scores=cov.mahalanobis(sc.transform(Xi))
    int_p95=np.percentile(int_scores,95)   # interictal 95th pct = natural alarm level
    for szid in uniq:
        pre=cov.mahalanobis(sc.transform(Xp[sid_p==szid]))
        # share of this seizure's preictal windows that clear the interictal 95th percentile
        detectable=float(np.mean(pre>int_p95))
        records.append({'patient':p,'seizure':szid,
                        'pre_max':float(pre.max()),'int_p95':float(int_p95),
                        'sep_ratio':float(pre.max()/int_p95),
                        'detectable_frac':detectable})

json.dump(records,open(f'{OUT}/ceiling_analysis.json','w'),indent=2)
sep=np.array([r['sep_ratio'] for r in records])
det=np.array([r['detectable_frac'] for r in records])
print(f"Total seizures analyzed: {len(records)}")
print(f"Seizures where preictal max EXCEEDS interictal-95 (sep_ratio>1): {np.mean(sep>1)*100:.1f}%")
print(f"Seizures essentially undetectable (detectable_frac<0.05): {np.mean(det<0.05)*100:.1f}%")
print(f"sep_ratio: median {np.median(sep):.2f}, 25th {np.percentile(sep,25):.2f}, 75th {np.percentile(sep,75):.2f}")

# distribution of separability across every seizure in the cohort
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].hist(sep,bins=30,color='#c0392b',edgecolor='white')
ax[0].axvline(1.0,color='black',ls='--',lw=1.5,label='detectability threshold')
ax[0].set_xlabel('Separability ratio (preictal max / interictal 95th pct)')
ax[0].set_ylabel('Number of seizures'); ax[0].set_title('Per-seizure detectability'); ax[0].legend()
ax[1].hist(det,bins=30,color='#2c3e50',edgecolor='white')
ax[1].set_xlabel('Fraction of preictal windows above interictal-95')
ax[1].set_ylabel('Number of seizures'); ax[1].set_title('Detectable preictal mass per seizure')
plt.tight_layout()
plt.savefig(f'{OUT}/figures/ceiling_separability.png',dpi=300,bbox_inches='tight')
plt.savefig(f'{OUT}/figures/ceiling_separability.pdf',bbox_inches='tight')
print(f"\nSaved figure: {OUT}/figures/ceiling_separability.png (+.pdf)")

In [ ]:
import json, numpy as np, os
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EmpiricalCovariance
np.random.seed(42)
OUT='/kaggle/working/artifacts'
cohort=json.load(open(f'{OUT}/cohort.json'))['cohort']
def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id'],d['seg_type']

def run_interictal_calibrated(fa_target):
    """Set tau from the interictal distribution instead of from the seizures.

    Interictal data is plentiful, so the threshold is stable. We pick tau to
    hold the false-alarm rate at fa_target, then read off whatever seizure-level
    sensitivity falls out. The interictal pool is split so the rate is measured
    on windows that did not set the threshold."""
    sens_list, fa_list = [], []
    for p in cohort:
        X,y,sid,seg=load_patient(p)
        Xp,Xi=X[y==1],X[y==0]; sid_p=sid[y==1]
        uniq=sorted(set(sid_p.tolist()))
        if len(uniq)<3: continue
        # one half sets tau, the other half measures the realised false-alarm rate
        idx=np.random.permutation(len(Xi)); cut=len(Xi)//2
        Xi_cal, Xi_test = Xi[idx[:cut]], Xi[idx[cut:]]
        sc=StandardScaler().fit(Xi_cal)
        cov=EmpiricalCovariance().fit(sc.transform(Xi_cal))
        cal_int_scores=cov.mahalanobis(sc.transform(Xi_cal))
        # tau as a percentile of the calibration scores
        tau=np.percentile(cal_int_scores, 100*(1-fa_target))
        # a seizure counts as caught if any of its preictal windows clears tau
        caught=[]
        for szid in uniq:
            pre=cov.mahalanobis(sc.transform(Xp[sid_p==szid]))
            caught.append(int(pre.max()>=tau))
        # realised false-alarm rate on the untouched half
        fa=np.mean(cov.mahalanobis(sc.transform(Xi_test))>=tau)
        sens_list.append(np.mean(caught)); fa_list.append(fa)
    return sens_list, fa_list

print("Interictal-calibrated thresholding (controls false-alarm rate):")
print(f"{'FA_target':<12}{'mean_sens':<12}{'worst':<10}{'meets>=0.9':<12}{'realized_FA'}")
for fa_t in [0.01, 0.02, 0.05, 0.10, 0.15]:
    sens,fa=run_interictal_calibrated(fa_t)
    meets=sum(s>=0.90-0.001 for s in sens)
    print(f"{fa_t:<12}{np.mean(sens):<12.3f}{np.min(sens):<10.3f}{f'{meets}/{len(sens)}':<12}{np.mean(fa):.3f}")

In [ ]:
import json, numpy as np, os
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EmpiricalCovariance
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

OUT='/kaggle/working/artifacts'; os.makedirs(f'{OUT}/figures',exist_ok=True)
cohort=json.load(open(f'{OUT}/cohort.json'))['cohort']
def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id'],d['seg_type']

def one_run(fa_target, seed):
    rng=np.random.RandomState(seed)
    per_pt={}
    for p in cohort:
        X,y,sid,seg=load_patient(p)
        Xp,Xi=X[y==1],X[y==0]; sid_p=sid[y==1]
        uniq=sorted(set(sid_p.tolist()))
        if len(uniq)<3: continue
        idx=rng.permutation(len(Xi)); cut=len(Xi)//2
        Xi_cal,Xi_test=Xi[idx[:cut]],Xi[idx[cut:]]
        sc=StandardScaler().fit(Xi_cal)
        cov=EmpiricalCovariance().fit(sc.transform(Xi_cal))
        tau=np.percentile(cov.mahalanobis(sc.transform(Xi_cal)),100*(1-fa_target))
        caught=[int(cov.mahalanobis(sc.transform(Xp[sid_p==s])).max()>=tau) for s in uniq]
        fa=np.mean(cov.mahalanobis(sc.transform(Xi_test))>=tau)
        per_pt[p]={'sens':float(np.mean(caught)),'fa':float(fa)}
    return per_pt

SEEDS=list(range(10))
FA_GRID=[0.01,0.02,0.05,0.10,0.15]

# Step 5 - sweep the false-alarm budget over ten random interictal splits,
# which gives the tradeoff curve and its error bars.
curve={}
for fa_t in FA_GRID:
    runs=[one_run(fa_t,s) for s in SEEDS]
    mean_sens=[np.mean([r[p]['sens'] for p in r]) for r in runs]
    real_fa=[np.mean([r[p]['fa'] for p in r]) for r in runs]
    meets=[sum(r[p]['sens']>=0.90-1e-9 for p in r) for r in runs]
    n_pt=len(runs[0])
    curve[fa_t]={'sens_mean':float(np.mean(mean_sens)),'sens_std':float(np.std(mean_sens)),
                 'fa_mean':float(np.mean(real_fa)),'fa_std':float(np.std(real_fa)),
                 'meets_mean':float(np.mean(meets)),'n_pt':n_pt}
    print(f"FA={fa_t}: sens {np.mean(mean_sens):.3f}±{np.std(mean_sens):.3f}  "
          f"realFA {np.mean(real_fa):.3f}±{np.std(real_fa):.3f}  meets {np.mean(meets):.1f}/{n_pt}")

json.dump(curve,open(f'{OUT}/multiseed_curve.json','w'),indent=2)

# ---- Figure 1: sensitivity against realised false-alarm rate ----
fa_x=[curve[f]['fa_mean'] for f in FA_GRID]
se_y=[curve[f]['sens_mean'] for f in FA_GRID]
se_e=[curve[f]['sens_std'] for f in FA_GRID]
fig,ax=plt.subplots(figsize=(6,4.2))
ax.errorbar(fa_x,se_y,yerr=se_e,marker='o',color='#c0392b',capsize=3,lw=2)
ax.axhline(0.90,ls='--',color='black',lw=1,label='target sensitivity 0.90')
ax.set_xlabel('Realized false-alarm rate'); ax.set_ylabel('Mean per-patient sensitivity')
ax.set_title('Sensitivity–false alarm tradeoff (10 seeds)'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}/figures/fig1_tradeoff.png',dpi=300,bbox_inches='tight')
plt.savefig(f'{OUT}/figures/fig1_tradeoff.pdf',bbox_inches='tight')

# ---- Figure 2: where each patient lands, averaged over the ten splits ----
runs05=[one_run(0.05,s) for s in SEEDS]
pts=sorted(runs05[0].keys())
psens=[np.mean([r[p]['sens'] for r in runs05]) for p in pts]
order=np.argsort(psens)
fig,ax=plt.subplots(figsize=(8,4.2))
colors=['#27ae60' if psens[i]>=0.90 else '#c0392b' for i in order]
ax.bar(range(len(pts)),[psens[i] for i in order],color=colors,edgecolor='white')
ax.axhline(0.90,ls='--',color='black',lw=1.2,label='guarantee 0.90')
ax.set_xticks(range(len(pts))); ax.set_xticklabels([pts[i] for i in order],rotation=90,fontsize=7)
ax.set_ylabel('Per-patient sensitivity'); ax.set_title('Per-patient guarantee (FA=0.05, green=met)')
ax.legend(); plt.tight_layout()
plt.savefig(f'{OUT}/figures/fig2_perpatient.png',dpi=300,bbox_inches='tight')
plt.savefig(f'{OUT}/figures/fig2_perpatient.pdf',bbox_inches='tight')

import shutil
shutil.make_archive(f'{OUT}/all_figures','zip',f'{OUT}/figures')
print("\nSaved fig1_tradeoff, fig2_perpatient (PNG+PDF) and all_figures.zip")

In [ ]:
import json, numpy as np, os
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EmpiricalCovariance
from scipy.stats import pearsonr
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

OUT='/kaggle/working/artifacts'
cohort=json.load(open(f'{OUT}/cohort.json'))['cohort']
def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id'],d['seg_type']

# Step 6 - can we tell in advance which patients will fail? Compare each
# patient's sensitivity against an index built only from calibration data:
# no test labels go into the index, so it is available before deployment.
SEEDS=list(range(10))
def patient_metrics(p):
    X,y,sid,seg=load_patient(p)
    Xp,Xi=X[y==1],X[y==0]; sid_p=sid[y==1]
    uniq=sorted(set(sid_p.tolist()))
    if len(uniq)<3: return None
    sens_runs=[]
    for s in SEEDS:
        rng=np.random.RandomState(s)
        idx=rng.permutation(len(Xi)); cut=len(Xi)//2
        Xi_cal=Xi[idx[:cut]]
        sc=StandardScaler().fit(Xi_cal)
        cov=EmpiricalCovariance().fit(sc.transform(Xi_cal))
        tau=np.percentile(cov.mahalanobis(sc.transform(Xi_cal)),95)  # FA target 0.05
        caught=[int(cov.mahalanobis(sc.transform(Xp[sid_p==z])).max()>=tau) for z in uniq]
        sens_runs.append(np.mean(caught))
    sens=float(np.mean(sens_runs))

    # the index itself: median separability ratio across a patient's seizures
    sc=StandardScaler().fit(Xi); cov=EmpiricalCovariance().fit(sc.transform(Xi))
    int95=np.percentile(cov.mahalanobis(sc.transform(Xi)),95)
    seps=[cov.mahalanobis(sc.transform(Xp[sid_p==z])).max()/int95 for z in uniq]
    det_index=float(np.median(seps))
    frac_undetectable=float(np.mean([s<1 for s in seps]))
    return {'patient':p,'sens':sens,'det_index':det_index,
            'frac_undetectable':frac_undetectable,'n_sz':len(uniq)}

rows=[patient_metrics(p) for p in cohort]
rows=[r for r in rows if r]
json.dump(rows,open(f'{OUT}/failure_mechanism.json','w'),indent=2)

sens=np.array([r['sens'] for r in rows])
det=np.array([r['det_index'] for r in rows])
und=np.array([r['frac_undetectable'] for r in rows])

r1,p1=pearsonr(det,sens)
r2,p2=pearsonr(und,sens)
print(f"Patients: {len(rows)}")
print(f"Corr(detectability_index, sensitivity): r={r1:.3f}, p={p1:.4f}")
print(f"Corr(frac_undetectable, sensitivity):   r={r2:.3f}, p={p2:.4f}")
print(f"\nPatients MISSING guarantee (sens<0.9): {[r['patient'] for r in rows if r['sens']<0.9]}")
print(f"Their mean frac_undetectable: {np.mean([r['frac_undetectable'] for r in rows if r['sens']<0.9]):.3f}")
print(f"Passing patients mean frac_undetectable: {np.mean([r['frac_undetectable'] for r in rows if r['sens']>=0.9]):.3f}")

# ---- Figure 3: index against realised sensitivity ----
fig,ax=plt.subplots(figsize=(6,4.2))
colors=['#27ae60' if s>=0.9 else '#c0392b' for s in sens]
ax.scatter(det,sens,c=colors,s=60,edgecolor='black',zorder=3)
ax.axhline(0.9,ls='--',color='black',lw=1,label='guarantee 0.90')
ax.axvline(1.0,ls=':',color='gray',lw=1)
ax.set_xlabel('Detectability index (median seizure separability, precomputable)')
ax.set_ylabel('Per-patient sensitivity (FA=0.05)')
ax.set_title(f'Detectability predicts guarantee success (r={r1:.2f})')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f'{OUT}/figures/fig3_failure_predictor.png',dpi=300,bbox_inches='tight')
plt.savefig(f'{OUT}/figures/fig3_failure_predictor.pdf',bbox_inches='tight')
import shutil; shutil.make_archive(f'{OUT}/all_figures','zip',f'{OUT}/figures')
print("\nSaved fig3_failure_predictor (PNG+PDF), updated all_figures.zip")

In [30]:
import os, glob

# locate the Siena mount
found=False
for root, dirs, files in os.walk('/kaggle/input'):
    if 'siena' in root.lower():
        edf=[f for f in files if f.endswith('.edf')]
        txt=[f for f in files if f.endswith('.txt')]
        if edf or txt:
            print("PATH:", root)
            print("  EDF:", len(edf), edf[:3])
            print("  TXT:", txt[:6])
            print()
            found=True

# print one annotation file too, since the format differs from CHB-MIT
print("="*50)
for tf in glob.glob('/kaggle/input/**/*eizure*.txt', recursive=True)[:1]:
    print("ANNOTATION SAMPLE:", tf)
    print(open(tf).read()[:1500])

PATH: /kaggle/input/datasets/atharvav75/physionet-siena-scalp-eeg-1-0-0-p1-p9/1.0.0
  EDF: 0 []
  TXT: ['SHA256SUMS.txt', 'LICENSE.txt']

PATH: /kaggle/input/datasets/atharvav75/physionet-siena-scalp-eeg-1-0-0-p1-p9/1.0.0/PN05
  EDF: 3 ['PN05-2.edf', 'PN05-4.edf', 'PN05-3.edf']
  TXT: ['Seizures-list-PN05.txt']

PATH: /kaggle/input/datasets/atharvav75/physionet-siena-scalp-eeg-1-0-0-p1-p9/1.0.0/PN00
  EDF: 5 ['PN00-1.edf', 'PN00-4.edf', 'PN00-5.edf']
  TXT: ['Seizures-list-PN00.txt']

PATH: /kaggle/input/datasets/atharvav75/physionet-siena-scalp-eeg-1-0-0-p1-p9/1.0.0/PN07
  EDF: 1 ['PN07-1.edf']
  TXT: ['Seizures-list-PN07.txt']

PATH: /kaggle/input/datasets/atharvav75/physionet-siena-scalp-eeg-1-0-0-p1-p9/1.0.0/PN01
  EDF: 1 ['PN01-1.edf']
  TXT: ['Seizures-list-PN01.txt']

PATH: /kaggle/input/datasets/atharvav75/physionet-siena-scalp-eeg-1-0-0-p1-p9/1.0.0/PN03
  EDF: 2 ['PN03-1.edf', 'PN03-2.edf']
  TXT: ['Seizures-list-PN03.txt']

PATH: /kaggle/input/datasets/atharvav75/physionet-si

In [31]:
import os, re, glob

SIENA_BASE='/kaggle/input/datasets/atharvav75/physionet-siena-scalp-eeg-1-0-0-p1-p9/1.0.0'

def hhmmss_to_sec(t):
    """Clock string such as '08.45.25' to seconds past midnight."""
    t=t.strip().replace(' ','')
    parts=re.split(r'[.:]',t)
    h,m,s=int(parts[0]),int(parts[1]),int(parts[2])
    return h*3600+m*60+s

def parse_siena(patient):
    """Seizure onsets and offsets per file, measured from that file's start."""
    txt_path=os.path.join(SIENA_BASE,patient,f'Seizures-list-{patient}.txt')
    text=open(txt_path).read()
    records=[]
    # split into seizure blocks
    blocks=re.split(r'Seizure n',text)
    for block in blocks[1:]:
        fn=re.search(r'File name:\s*(\S+\.edf)',block)
        reg=re.search(r'Registration start time:\s*([\d.:]+)',block)
        sz_s=re.search(r'Seizure start time:\s*([\d.:]+)',block)
        sz_e=re.search(r'Seizure end time:\s*([\d.:]+)',block)
        if not (fn and reg and sz_s and sz_e): continue
        reg_sec=hhmmss_to_sec(reg.group(1))
        onset=hhmmss_to_sec(sz_s.group(1))-reg_sec
        offset=hhmmss_to_sec(sz_e.group(1))-reg_sec
        # recordings that run past midnight come out negative; wrap them
        if onset<0: onset+=24*3600; offset+=24*3600
        records.append({'file':fn.group(1),'onset':onset,'offset':offset})
    return records

# verify on PN05
recs=parse_siena('PN05')
print("PN05 seizures:")
for r in recs:
    print(f"  {r['file']}: onset={r['onset']}s ({r['onset']//60}min), offset={r['offset']}s")

# inventory all patients
print("\n"+"="*50)
patients=sorted([d for d in os.listdir(SIENA_BASE) if d.startswith('PN')])
print(f"{'Patient':<10}{'Seizures':<10}{'Files w/ sz'}")
print("-"*35)
total_eligible=[]
for p in patients:
    try: recs=parse_siena(p)
    except Exception as e:
        print(f"{p:<10}ERROR: {e}"); continue
    nf=len(set(r['file'] for r in recs))
    print(f"{p:<10}{len(recs):<10}{nf}")
    if len(recs)>=3: total_eligible.append(p)
print("-"*35)
print(f"Eligible (>=3 seizures): {total_eligible}")

PN05 seizures:
  PN05-2.edf: onset=7163s (119min), offset=7198s
  PN05-3.edf: onset=6836s (113min), offset=6866s
  PN05-4.edf: onset=3608s (60min), offset=3647s

Patient   Seizures  Files w/ sz
-----------------------------------
PN00      5         5
PN01      0         0
PN03      2         2
PN05      3         3
PN06      5         5
PN07      1         1
PN09      3         3
PN10      ERROR: list index out of range
-----------------------------------
Eligible (>=3 seizures): ['PN00', 'PN05', 'PN06', 'PN09']


In [32]:
txt=open(f'{SIENA_BASE}/PN10/Seizures-list-PN10.txt').read()
print(txt[:2000])

PN10  

Data Sampling Rate: 512 Hz

Channels in EDF files:
Channel 1: Fp1      
Channel 2: F3      
Channel 3: C3      
Channel 4: P3      
Channel 5: 1      
Channel 6: F7      
Channel 7: T3      
Channel 8: T5      
Channel 14: Fz      
Channel 15: Cz      
Channel 16: Pz      
Channel 17: F4      
Channel 18: C4      
Channel 19: P4      
Channel 20: O2      
Channel 21: F8      
Channel 22: T4       
Channel 23: T6      
Channel 32: EKG 1    
Channel 33: EKG 2    
Channel 34: Fp2



Seizure n 1 (seizure 1A):
File name: PN10-1.edf
Registration start time: 05.40.05
Registration end time: 08.26.07
Seizure start time: 07.45.50 
Seizure end time: 07.46.59



Seizure n 2 (seizure B1):
File name: PN10-2.edf
Registration start time: 09.30.15
Registration end time: 11.52.29
Seizure start time: 11.40.13
Seizure end time: 11.41.04 opure 11.40.43



Seizure n 3 (seizure B2):
File name: PN10-3.edf
Registration start time: 13.33.18
Registration end time: 15.58.31
Seizure start time: 15.43.53 (C

In [33]:
import os, re

SIENA_BASE='/kaggle/input/datasets/atharvav75/physionet-siena-scalp-eeg-1-0-0-p1-p9/1.0.0'

def hhmmss_to_sec(t):
    t=t.strip()
    # take the first well-formed HH.MM.SS and ignore whatever trails it
    m=re.search(r'(\d{1,2})\s*[.:]\s*(\d{2})\s*[.:]\s*(\d{2})', t)
    if not m: return None
    h,mn,s=int(m.group(1)),int(m.group(2)),int(m.group(3))
    return h*3600+mn*60+s

def parse_siena(patient):
    txt_path=os.path.join(SIENA_BASE,patient,f'Seizures-list-{patient}.txt')
    text=open(txt_path).read()
    records=[]
    blocks=re.split(r'Seizure n',text)
    for block in blocks[1:]:
        fn=re.search(r'File name:\s*(\S+\.edf)',block)
        reg=re.search(r'Registration start time:\s*([^\n]+)',block)
        sz_s=re.search(r'Seizure start time:\s*([^\n]+)',block)
        sz_e=re.search(r'Seizure end time:\s*([^\n]+)',block)
        if not (fn and reg and sz_s and sz_e): continue
        reg_sec=hhmmss_to_sec(reg.group(1))
        onset=hhmmss_to_sec(sz_s.group(1))   # takes FIRST time = clinical onset
        offset=hhmmss_to_sec(sz_e.group(1))
        if None in (reg_sec,onset,offset): continue
        onset-=reg_sec; offset-=reg_sec
        if onset<0: onset+=24*3600; offset+=24*3600
        if offset<onset: offset+=24*3600   # offset crossing midnight
        records.append({'file':fn.group(1),'onset':onset,'offset':offset})
    return records

# re-inventory all
patients=sorted([d for d in os.listdir(SIENA_BASE) if d.startswith('PN')])
print(f"{'Patient':<10}{'Seizures':<10}{'Files w/ sz'}")
print("-"*35)
eligible=[]
for p in patients:
    try: recs=parse_siena(p)
    except Exception as e:
        print(f"{p:<10}ERROR: {e}"); continue
    nf=len(set(r['file'] for r in recs))
    print(f"{p:<10}{len(recs):<10}{nf}")
    if len(recs)>=3: eligible.append(p)
print("-"*35)
print(f"Eligible (>=3 seizures): {eligible}")

# PN10 onsets should land tens of minutes in; anything else means a parse slip
print("\nPN10 detail:")
for r in parse_siena('PN10'):
    print(f"  {r['file']}: onset={r['onset']}s ({r['onset']//60}min), dur={r['offset']-r['onset']}s")

Patient   Seizures  Files w/ sz
-----------------------------------
PN00      5         5
PN01      0         0
PN03      2         2
PN05      3         3
PN06      5         5
PN07      1         1
PN09      3         3
PN10      10        6
-----------------------------------
Eligible (>=3 seizures): ['PN00', 'PN05', 'PN06', 'PN09', 'PN10']

PN10 detail:
  PN10-1.edf: onset=7545s (125min), dur=69s
  PN10-2.edf: onset=7798s (129min), dur=51s
  PN10-3.edf: onset=7835s (130min), dur=69s
  PN10-4.5.6.edf: onset=2309s (38min), dur=5s
  PN10-4.5.6.edf: onset=6544s (109min), dur=19s
  PN10-4.5.6.edf: onset=11225s (187min), dur=57s
  PN10-7.8.9.edf: onset=38748s (645min), dur=48s
  PN10-7.8.9.edf: onset=5459s (90min), dur=18s
  PN10-7.8.9.edf: onset=12923s (215min), dur=15s
  PN10-10.edf: onset=7977s (132min), dur=14s


In [34]:
import mne
mne.set_log_level('ERROR')

def parse_siena_validated(patient):
    """Parse, then drop any seizure whose onset falls outside the recording."""
    recs=parse_siena(patient)
    valid=[]
    for r in recs:
        fpath=os.path.join(SIENA_BASE,patient,r['file'])
        try:
            raw=mne.io.read_raw_edf(fpath,preload=False)
            dur=raw.n_times/raw.info['sfreq']
        except Exception:
            continue
        # the full 35 min run-up has to fit inside the file or the seizure is unusable
        if r['onset']-35*60 >= 0 and r['onset'] <= dur:
            r['file_dur']=dur; valid.append(r)
    return valid

print(f"{'Patient':<10}{'Raw sz':<9}{'Valid sz':<10}{'Eligible'}")
print("-"*40)
eligible_v=[]
for p in ['PN00','PN05','PN06','PN09','PN10']:
    raw_n=len(parse_siena(p))
    valid=parse_siena_validated(p)
    elig=len(valid)>=3
    if elig: eligible_v.append(p)
    print(f"{p:<10}{raw_n:<9}{len(valid):<10}{'YES' if elig else 'no'}")
print("-"*40)
print(f"Validated eligible cohort: {eligible_v}")

Patient   Raw sz   Valid sz  Eligible
----------------------------------------
PN00      5        0         no
PN05      3        3         YES
PN06      5        2         no
PN09      3        3         YES
PN10      10       3         YES
----------------------------------------
Validated eligible cohort: ['PN05', 'PN09', 'PN10']


In [35]:
# PN00 lost every seizure - compare annotated onsets against file durations
import mne; mne.set_log_level('ERROR')
for p in ['PN00','PN06']:
    print(f"\n{p}:")
    for r in parse_siena(p):
        fpath=os.path.join(SIENA_BASE,p,r['file'])
        try:
            raw=mne.io.read_raw_edf(fpath,preload=False)
            dur=raw.n_times/raw.info['sfreq']
        except Exception as e:
            print(f"  {r['file']}: LOAD FAIL {e}"); continue
        fits = (r['onset']-35*60>=0) and (r['onset']<=dur)
        print(f"  {r['file']}: onset={r['onset']}s ({r['onset']//60}min), file_dur={dur:.0f}s ({dur//60:.0f}min), preictal_fits={fits}")


PN00:
  PN00-1.edf: onset=1143s (19min), file_dur=2625s (43min), preictal_fits=False
  PN00-2.edf: onset=1220s (20min), file_dur=2302s (38min), preictal_fits=False
  PN00-3.edf: onset=765s (12min), file_dur=2509s (41min), preictal_fits=False
  PN00-4.edf: onset=1006s (16min), file_dur=2102s (35min), preictal_fits=False
  PN00-5.edf: onset=904s (15min), file_dur=2143s (35min), preictal_fits=False

PN06:
  PNO6-1.edf: LOAD FAIL File does not exist: "/kaggle/input/datasets/atharvav75/physionet-siena-scalp-eeg-1-0-0-p1-p9/1.0.0/PN06/PNO6-1.edf"
  PNO6-2.edf: LOAD FAIL File does not exist: "/kaggle/input/datasets/atharvav75/physionet-siena-scalp-eeg-1-0-0-p1-p9/1.0.0/PN06/PNO6-2.edf"
  PN06-3.edf: onset=6275s (104min), file_dur=8180s (136min), preictal_fits=True
  PNO6-4.edf: LOAD FAIL File does not exist: "/kaggle/input/datasets/atharvav75/physionet-siena-scalp-eeg-1-0-0-p1-p9/1.0.0/PN06/PNO6-4.edf"
  PN06-5.edf: onset=4783s (79min), file_dur=6022s (100min), preictal_fits=True


In [36]:
def fix_filename(patient, fname):
    """Some Siena annotations write the letter O where the folder uses a zero."""
    import os
    pdir=os.path.join(SIENA_BASE,patient)
    if os.path.exists(os.path.join(pdir,fname)):
        return fname
    # try swapping letter-O for zero and vice versa in the PN prefix
    cand=fname.replace('PNO','PN0').replace('PN0','PN0')
    # brute: match against actual files case-insensitively on O/0
    for actual in os.listdir(pdir):
        if actual.lower().replace('o','0')==fname.lower().replace('o','0'):
            return actual
    return fname

def parse_siena_validated_v2(patient):
    import mne; mne.set_log_level('ERROR')
    recs=parse_siena(patient)
    valid=[]
    for r in recs:
        real=fix_filename(patient,r['file'])
        fpath=os.path.join(SIENA_BASE,patient,real)
        try:
            raw=mne.io.read_raw_edf(fpath,preload=False); dur=raw.n_times/raw.info['sfreq']
        except Exception: continue
        if r['onset']-35*60>=0 and r['onset']<=dur:
            r['file']=real; r['file_dur']=dur; valid.append(r)
    return valid

print(f"{'Patient':<10}{'Valid sz':<10}{'Eligible'}")
print("-"*30)
elig=[]
for p in ['PN00','PN05','PN06','PN09','PN10']:
    v=parse_siena_validated_v2(p)
    e=len(v)>=3
    if e: elig.append(p)
    print(f"{p:<10}{len(v):<10}{'YES' if e else 'no'}")
print(f"\nFinal Siena cohort: {elig}")

Patient   Valid sz  Eligible
------------------------------
PN00      0         no
PN05      3         YES
PN06      5         YES
PN09      3         YES
PN10      3         YES

Final Siena cohort: ['PN05', 'PN06', 'PN09', 'PN10']


In [37]:
import os, numpy as np, mne
from scipy import signal as sps
from scipy.stats import entropy as scipy_entropy
mne.set_log_level('ERROR')

OUT='/kaggle/working/artifacts'
SIENA_COHORT=['PN05','PN06','PN09','PN10']
SPH_MIN,SOP_MIN,INTERICTAL_GAP_H=5,30,4
WIN_SEC,FS=30,256   # SAME as CHB-MIT: resample Siena 512->256
BANDS={'delta':(0.5,4),'theta':(4,8),'alpha':(8,13),'beta':(13,30),'gamma':(30,50)}

def extract_features(win_data,fs=FS):   # IDENTICAL to CHB-MIT
    feats=[]
    for ch in win_data:
        f,pxx=sps.welch(ch,fs=fs,nperseg=min(256,len(ch)))
        total=np.trapezoid(pxx,f)+1e-12
        for (lo,hi) in BANDS.values():
            idx=(f>=lo)&(f<hi); feats.append(np.trapezoid(pxx[idx],f[idx])/total)
        feats.append(np.sum(np.abs(np.diff(ch))))
        p=pxx/(pxx.sum()+1e-12); feats.append(scipy_entropy(p+1e-12))
    return np.array(feats,dtype=np.float32)

def siena_common_channels(patient):
    """Channels common to all of a patient's files, minus EKG and status lines."""
    valid=parse_siena_validated_v2(patient)
    files=set(r['file'] for r in valid)
    # also include seizure-free files for interictal
    all_edf=[f for f in os.listdir(os.path.join(SIENA_BASE,patient)) if f.endswith('.edf')]
    ch_sets=[]
    for f in all_edf:
        try:
            raw=mne.io.read_raw_edf(os.path.join(SIENA_BASE,patient,f),preload=False)
            ch_sets.append(set(raw.ch_names))
        except Exception: continue
    if not ch_sets: return []
    common=set.intersection(*ch_sets)
    # keep EEG channels only: drop EKG, numeric-only, status
    common=sorted([c for c in common if not any(x in c.upper() for x in ['EKG','ECG','STATUS','MARK']) and not c.strip().isdigit()])
    return common

def label_file(onset_list, dur):
    """Window labelling, reusing the CHB-MIT timing rules unchanged."""
    labels=[]; n=int(dur//WIN_SEC)
    for w in range(n):
        ws,we=w*WIN_SEC,(w+1)*WIN_SEC; wc=(ws+we)/2; lab='interictal_candidate'
        for (sz_s,sz_e) in onset_list:
            pre_s=sz_s-(SPH_MIN+SOP_MIN)*60; pre_e=sz_s-SPH_MIN*60
            if pre_s<=wc<pre_e: lab='preictal'; break
            if pre_e<=wc<sz_s: lab='discard'; break
            if sz_s<=wc<sz_e+INTERICTAL_GAP_H*3600: lab='discard'; break
            if pre_s-INTERICTAL_GAP_H*3600<=wc<pre_s: lab='discard'; break
        labels.append((ws,we,lab))
    return labels

def process_siena_patient(patient):
    valid=parse_siena_validated_v2(patient)
    chans=siena_common_channels(patient)
    if len(chans)<16: return None
    # group seizures by file
    from collections import defaultdict
    sz_by_file=defaultdict(list)
    for r in valid: sz_by_file[r['file']].append((r['onset'],r['offset']))
    all_edf=[f for f in os.listdir(os.path.join(SIENA_BASE,patient)) if f.endswith('.edf')]
    sz_files=set(sz_by_file.keys())

    X,y,sid=[],[],[]; sz_counter=0
    for f in all_edf:
        fpath=os.path.join(SIENA_BASE,patient,f)
        try: raw=mne.io.read_raw_edf(fpath,preload=True)
        except Exception: continue
        raw.pick_channels(chans,ordered=True)
        if abs(raw.info['sfreq']-FS)>1: raw.resample(FS)   # 512->256
        raw.filter(0.5,50.,verbose='ERROR')
        try: raw.notch_filter(50.,verbose='ERROR')   # Europe = 50Hz mains
        except Exception: pass
        data=raw.get_data(); dur=data.shape[1]/FS
        onsets=sz_by_file.get(f,[])
        # interictal only from seizure-free files
        is_clean=(f not in sz_files)
        labels=label_file(onsets,dur)
        for (ws,we,lab) in labels:
            s0,s1=int(ws*FS),int(we*FS)
            if s1>data.shape[1]: continue
            if lab=='preictal':
                X.append(extract_features(data[:,s0:s1])); y.append(1); sid.append(f'{patient}_sz{sz_counter}')
            elif lab=='interictal_candidate' and is_clean:
                X.append(extract_features(data[:,s0:s1])); y.append(0); sid.append(f'{patient}_inter')
        if onsets: sz_counter+=len(onsets)
    if not X or sum(y)==0: return None
    return np.array(X),np.array(y),np.array(sid),chans

print("Processing Siena cohort...")
siena_report={}
for p in SIENA_COHORT:
    res=process_siena_patient(p)
    if res is None:
        siena_report[p]='SKIPPED'; print(f"  {p}: skipped"); continue
    X,y,sid,chans=res
    np.savez_compressed(f'{OUT}/SIENA_{p}_features.npz',X=X,y=y,seizure_id=sid)
    npre,nint=int((y==1).sum()),int((y==0).sum())
    nsz=len(set(s for s in sid if 'sz' in s))
    siena_report[p]={'n_ch':len(chans),'n_pre':npre,'n_int':nint,'n_sz':nsz}
    print(f"  {p}: {len(chans)}ch, pre={npre}, int={nint}, seizures={nsz}")

import json
json.dump(siena_report,open(f'{OUT}/siena_report.json','w'),indent=2)
print("\nSaved Siena features + report")

Processing Siena cohort...
  PN05: 34ch, pre=180, int=0, seizures=3
  PN06: 34ch, pre=300, int=0, seizures=5
  PN09: 38ch, pre=180, int=0, seizures=3
  PN10: 38ch, pre=180, int=0, seizures=3

Saved Siena features + report


In [39]:
def label_file_v2(onset_list, dur):
    """Relaxed labelling for Siena: interictal taken from inside seizure files.

    Siena recordings are too short to spare whole seizure-free files, so a
    window counts as interictal if it sits at least INTERICTAL_GAP_H away from
    every seizure in its own file."""
    labels=[]; n=int(dur//WIN_SEC)
    for w in range(n):
        ws,we=w*WIN_SEC,(w+1)*WIN_SEC; wc=(ws+we)/2; lab='interictal_candidate'
        for (sz_s,sz_e) in onset_list:
            pre_s=sz_s-(SPH_MIN+SOP_MIN)*60; pre_e=sz_s-SPH_MIN*60
            if pre_s<=wc<pre_e: lab='preictal'; break
            if pre_e<=wc<sz_s: lab='discard'; break
            if sz_s<=wc<sz_e+INTERICTAL_GAP_H*3600: lab='discard'; break
            if pre_s-INTERICTAL_GAP_H*3600<=wc<pre_s: lab='discard'; break
        # if still candidate, require >=4h from ALL seizures in file (else discard)
        if lab=='interictal_candidate':
            for (sz_s,sz_e) in onset_list:
                if abs(wc-sz_s)<INTERICTAL_GAP_H*3600 or abs(wc-sz_e)<INTERICTAL_GAP_H*3600:
                    lab='discard'; break
        labels.append((ws,we,lab))
    return labels

In [40]:
import os, numpy as np, mne, json
from scipy import signal as sps
from scipy.stats import entropy as scipy_entropy
from collections import defaultdict
mne.set_log_level('ERROR')

OUT='/kaggle/working/artifacts'
SIENA_COHORT=['PN05','PN06','PN09','PN10']
SPH_MIN,SOP_MIN=5,30
INTERICTAL_GAP_H=1.0     # relaxed from 4 h: Siena recordings are far shorter than CHB-MIT
WIN_SEC,FS=30,256
BANDS={'delta':(0.5,4),'theta':(4,8),'alpha':(8,13),'beta':(13,30),'gamma':(30,50)}

def extract_features(win_data,fs=FS):
    feats=[]
    for ch in win_data:
        f,pxx=sps.welch(ch,fs=fs,nperseg=min(256,len(ch)))
        total=np.trapezoid(pxx,f)+1e-12
        for (lo,hi) in BANDS.values():
            idx=(f>=lo)&(f<hi); feats.append(np.trapezoid(pxx[idx],f[idx])/total)
        feats.append(np.sum(np.abs(np.diff(ch))))
        p=pxx/(pxx.sum()+1e-12); feats.append(scipy_entropy(p+1e-12))
    return np.array(feats,dtype=np.float32)

def siena_common_channels(patient):
    all_edf=[f for f in os.listdir(os.path.join(SIENA_BASE,patient)) if f.endswith('.edf')]
    ch_sets=[]
    for f in all_edf:
        try:
            raw=mne.io.read_raw_edf(os.path.join(SIENA_BASE,patient,f),preload=False)
            ch_sets.append(set(raw.ch_names))
        except Exception: continue
    if not ch_sets: return []
    common=set.intersection(*ch_sets)
    common=sorted([c for c in common if not any(x in c.upper() for x in ['EKG','ECG','STATUS','MARK']) and not c.strip().isdigit()])
    return common

def label_file_v2(onset_list,dur):
    labels=[]; n=int(dur//WIN_SEC)
    for w in range(n):
        ws,we=w*WIN_SEC,(w+1)*WIN_SEC; wc=(ws+we)/2; lab='interictal_candidate'
        for (sz_s,sz_e) in onset_list:
            pre_s=sz_s-(SPH_MIN+SOP_MIN)*60; pre_e=sz_s-SPH_MIN*60
            if pre_s<=wc<pre_e: lab='preictal'; break
            if pre_e<=wc<sz_s: lab='discard'; break
            if sz_s<=wc<sz_e+INTERICTAL_GAP_H*3600: lab='discard'; break
            if pre_s-INTERICTAL_GAP_H*3600<=wc<pre_s: lab='discard'; break
        if lab=='interictal_candidate':
            for (sz_s,sz_e) in onset_list:
                if abs(wc-sz_s)<INTERICTAL_GAP_H*3600 or abs(wc-sz_e)<INTERICTAL_GAP_H*3600:
                    lab='discard'; break
        labels.append((ws,we,lab))
    return labels

def process_siena_patient(patient):
    valid=parse_siena_validated_v2(patient)
    chans=siena_common_channels(patient)
    if len(chans)<16: return None
    sz_by_file=defaultdict(list)
    for r in valid: sz_by_file[r['file']].append((r['onset'],r['offset']))
    all_edf=[f for f in os.listdir(os.path.join(SIENA_BASE,patient)) if f.endswith('.edf')]
    X,y,sid=[],[],[]; sz_counter=0
    for f in all_edf:
        try: raw=mne.io.read_raw_edf(os.path.join(SIENA_BASE,patient,f),preload=True)
        except Exception: continue
        raw.pick_channels(chans,ordered=True)
        if abs(raw.info['sfreq']-FS)>1: raw.resample(FS)
        raw.filter(0.5,50.,verbose='ERROR')
        try: raw.notch_filter(50.,verbose='ERROR')
        except Exception: pass
        data=raw.get_data(); dur=data.shape[1]/FS
        onsets=sz_by_file.get(f,[])
        labels=label_file_v2(onsets,dur)
        for (ws,we,lab) in labels:
            s0,s1=int(ws*FS),int(we*FS)
            if s1>data.shape[1]: continue
            if lab=='preictal':
                X.append(extract_features(data[:,s0:s1])); y.append(1); sid.append(f'{patient}_sz{sz_counter}')
            elif lab=='interictal_candidate':
                X.append(extract_features(data[:,s0:s1])); y.append(0); sid.append(f'{patient}_inter')
        if onsets: sz_counter+=len(onsets)
    if not X or sum(y)==0: return None
    return np.array(X),np.array(y),np.array(sid),chans

print(f"Processing Siena (interictal gap = {INTERICTAL_GAP_H}h)...")
siena_report={}
for p in SIENA_COHORT:
    res=process_siena_patient(p)
    if res is None:
        siena_report[p]='SKIPPED'; print(f"  {p}: skipped"); continue
    X,y,sid,chans=res
    np.savez_compressed(f'{OUT}/SIENA_{p}_features.npz',X=X,y=y,seizure_id=sid)
    npre,nint=int((y==1).sum()),int((y==0).sum())
    nsz=len(set(s for s in sid if 'sz' in s))
    siena_report[p]={'n_ch':len(chans),'n_pre':npre,'n_int':nint,'n_sz':nsz}
    print(f"  {p}: {len(chans)}ch, pre={npre}, int={nint}, seizures={nsz}")
json.dump(siena_report,open(f'{OUT}/siena_report.json','w'),indent=2)
print("\nSaved.")

Processing Siena (interictal gap = 1.0h)...
  PN05: 34ch, pre=180, int=87, seizures=3
  PN06: 34ch, pre=300, int=146, seizures=5
  PN09: 38ch, pre=180, int=151, seizures=3
  PN10: 38ch, pre=180, int=207, seizures=3

Saved.


In [41]:
import json, numpy as np, os
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.covariance import EmpiricalCovariance
np.random.seed(42)
OUT='/kaggle/working/artifacts'

def load_npz(path):
    d=np.load(path,allow_pickle=True); return d['X'],d['y'],d['seizure_id']

PCA_DIM=15

def eval_cohort(patient_files, label):
    """Run the interictal-calibrated anomaly method on a cohort, per patient.

    Features are reduced with PCA first so the covariance stays well behaved on
    the smaller Siena recordings. Returns sensitivity at false-alarm targets of
    0.05 and 0.10, averaged over seeds."""
    results={}
    for p,path in patient_files:
        try: X,y,sid=load_npz(path)
        except Exception: continue
        Xp,Xi=X[y==1],X[y==0]; sid_p=sid[y==1]
        uniq=sorted(set(sid_p.tolist()))
        if len(uniq)<3 or len(Xi)<PCA_DIM*2: continue
        for fa_t in [0.05,0.10]:
            sens_seeds=[]
            for seed in range(10):
                rng=np.random.RandomState(seed)
                idx=rng.permutation(len(Xi)); cut=len(Xi)//2
                Xi_cal,Xi_test=Xi[idx[:cut]],Xi[idx[cut:]]
                if len(Xi_cal)<PCA_DIM+2: continue
                sc=StandardScaler().fit(Xi_cal)
                pca=PCA(n_components=min(PCA_DIM,Xi_cal.shape[1],len(Xi_cal)-1)).fit(sc.transform(Xi_cal))
                Zc=pca.transform(sc.transform(Xi_cal))
                cov=EmpiricalCovariance().fit(Zc)
                score=lambda A: cov.mahalanobis(pca.transform(sc.transform(A)))
                tau=np.percentile(score(Xi_cal),100*(1-fa_t))
                caught=[int(score(Xp[sid_p==z]).max()>=tau) for z in uniq]
                sens_seeds.append(np.mean(caught))
            if sens_seeds:
                results.setdefault(p,{})[f'sens_fa{fa_t}']=float(np.mean(sens_seeds))
    return results

# re-run CHB-MIT through the same PCA path so the two are comparable
chb_cohort=json.load(open(f'{OUT}/cohort.json'))['cohort']
chb_files=[(p,f'{OUT}/{p}_features.npz') for p in chb_cohort]
chb_res=eval_cohort(chb_files,'CHB-MIT')

# Siena
siena_files=[(p,f'{OUT}/SIENA_{p}_features.npz') for p in ['PN05','PN06','PN09','PN10']]
siena_res=eval_cohort(siena_files,'Siena')

def summarize(res,name):
    for fa in [0.05,0.10]:
        vals=[v[f'sens_fa{fa}'] for v in res.values() if f'sens_fa{fa}' in v]
        meets=sum(s>=0.9-1e-9 for s in vals)
        print(f"[{name} FA={fa}] N={len(vals)}  mean_sens={np.mean(vals):.3f}  worst={np.min(vals):.3f}  meets>=0.9: {meets}/{len(vals)}")

print("PCA-Mahalanobis, applied identically to both datasets:\n")
summarize(chb_res,"CHB-MIT")
print()
summarize(siena_res,"Siena")

json.dump({'chb':chb_res,'siena':siena_res},open(f'{OUT}/cross_dataset.json','w'),indent=2)

# only a handful of Siena patients survive, so list them individually
print("\nSiena per-patient (FA=0.05):")
for p,v in siena_res.items():
    print(f"  {p}: sens={v.get('sens_fa0.05','NA'):.3f}")

PCA-Mahalanobis, applied identically to both datasets:

[CHB-MIT FA=0.05] N=18  mean_sens=0.718  worst=0.314  meets>=0.9: 5/18
[CHB-MIT FA=0.1] N=18  mean_sens=0.871  worst=0.571  meets>=0.9: 11/18

[Siena FA=0.05] N=4  mean_sens=0.892  worst=0.567  meets>=0.9: 3/4
[Siena FA=0.1] N=4  mean_sens=0.925  worst=0.700  meets>=0.9: 3/4

Siena per-patient (FA=0.05):
  PN05: sens=1.000
  PN06: sens=1.000
  PN09: sens=0.567
  PN10: sens=1.000


In [42]:
import json, numpy as np, os
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EmpiricalCovariance
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
np.random.seed(42)
OUT='/kaggle/working/artifacts'; os.makedirs(f'{OUT}/figures',exist_ok=True)
cohort=json.load(open(f'{OUT}/cohort.json'))['cohort']

def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id']

# Step 7 - selective forecasting. Pair each patient's sensitivity with their
# detectability index D, using the full-dimensional score.
SEEDS=range(10)
rows=[]
for p in cohort:
    X,y,sid=load_patient(p)
    Xp,Xi=X[y==1],X[y==0]; sid_p=sid[y==1]
    uniq=sorted(set(sid_p.tolist()))
    if len(uniq)<3: continue
    sens_seeds=[]; D_seeds=[]
    for seed in SEEDS:
        rng=np.random.RandomState(seed)
        idx=rng.permutation(len(Xi)); cut=len(Xi)//2
        Xi_cal=Xi[idx[:cut]]
        sc=StandardScaler().fit(Xi_cal)
        cov=EmpiricalCovariance().fit(sc.transform(Xi_cal))
        score=lambda A: cov.mahalanobis(sc.transform(A))
        int_scores=score(Xi_cal); q95=np.percentile(int_scores,95)
        tau=q95  # FA target 0.05
        peak=[score(Xp[sid_p==z]).max() for z in uniq]
        caught=[int(pk>=tau) for pk in peak]
        sens_seeds.append(np.mean(caught))
        # D comes from the same calibration interictal data, so it is known up front
        D_seeds.append(np.mean([pk<q95 for pk in peak]))
    rows.append({'patient':p,'sens':float(np.mean(sens_seeds)),'D':float(np.mean(D_seeds))})

# ---- Refuse the patients whose D sits above a cutoff, and see what that buys ----
rows_sorted=sorted(rows,key=lambda r:r['D'])
print("Selective prediction: abstain on high-D patients\n")
print(f"{'D_thresh':<10}{'served':<9}{'abstain':<9}{'served_meets':<13}{'served_mean_sens':<16}{'coverage'}")
print("-"*70)
curve=[]
for Dt in [1.01,0.30,0.25,0.20,0.15,0.10,0.05,0.001]:
    served=[r for r in rows if r['D']<Dt]
    if not served: continue
    meets=sum(r['sens']>=0.9-1e-9 for r in served)
    guar_rate=meets/len(served)
    mean_s=np.mean([r['sens'] for r in served])
    cov_frac=len(served)/len(rows)
    curve.append({'Dt':Dt,'n_served':len(served),'guar_rate':guar_rate,'mean_sens':mean_s,'coverage':cov_frac})
    print(f"{Dt:<10}{len(served):<9}{len(rows)-len(served):<9}{f'{meets}/{len(served)}':<13}{mean_s:<16.3f}{cov_frac:.2f}")

json.dump({'rows':rows,'curve':curve},open(f'{OUT}/selective_prediction.json','w'),indent=2)

# baseline: serve everybody
base=curve[0]
print(f"\nNo abstention (serve all): {base['guar_rate']*100:.0f}% of patients meet guarantee")
# the loosest cutoff at which every served patient still meets the target
full=[c for c in curve if c['guar_rate']>=0.999]
if full:
    best=max(full,key=lambda c:c['coverage'])
    print(f"With abstention at D<{best['Dt']}: {best['guar_rate']*100:.0f}% guarantee met, "
          f"serving {best['coverage']*100:.0f}% of patients ({best['n_served']}/{len(rows)})")

# ---- Figure 4: reliability against coverage ----
cov=[c['coverage'] for c in curve]; gr=[c['guar_rate'] for c in curve]
fig,ax=plt.subplots(figsize=(6,4.2))
ax.plot(cov,gr,marker='o',color='#c0392b',lw=2)
ax.set_xlabel('Coverage (fraction of patients served)')
ax.set_ylabel('Guarantee satisfaction rate (served patients)')
ax.set_title('Selective forecasting: abstention raises served-patient reliability')
ax.axhline(1.0,ls='--',color='gray',lw=1)
ax.grid(alpha=0.3); ax.set_ylim(0.5,1.05); plt.tight_layout()
plt.savefig(f'{OUT}/figures/fig4_selective.png',dpi=300,bbox_inches='tight')
plt.savefig(f'{OUT}/figures/fig4_selective.pdf',bbox_inches='tight')
import shutil; shutil.make_archive(f'{OUT}/all_figures','zip',f'{OUT}/figures')
print("\nSaved fig4_selective (PNG+PDF), updated all_figures.zip")

Selective prediction: abstain on high-D patients

D_thresh  served   abstain  served_meets served_mean_senscoverage
----------------------------------------------------------------------
1.01      18       0        14/18        0.941           1.00
0.3       17       1        14/17        0.957           0.94
0.25      15       3        14/15        0.990           0.83
0.2       15       3        14/15        0.990           0.83
0.15      15       3        14/15        0.990           0.83
0.1       14       4        14/14        1.000           0.78
0.05      14       4        14/14        1.000           0.78
0.001     14       4        14/14        1.000           0.78

No abstention (serve all): 78% of patients meet guarantee
With abstention at D<0.1: 100% guarantee met, serving 78% of patients (14/18)

Saved fig4_selective (PNG+PDF), updated all_figures.zip


In [43]:
import json, numpy as np, os
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EmpiricalCovariance
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
np.random.seed(42)
OUT='/kaggle/working/artifacts'
cohort=json.load(open(f'{OUT}/cohort.json'))['cohort']

def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id']

def sens_under_split(p, split_type):
    """Split the seizures two ways to separate two possible causes of failure.

    A random split exposes plain seizure-to-seizure variation. A time-ordered
    split exposes that same variation plus any drift over the recording. The
    gap between them is what drift is worth."""
    X,y,sid=load_patient(p)
    Xp,Xi=X[y==1],X[y==0]; sid_p=sid[y==1]
    uniq=sorted(set(sid_p.tolist()))          # sorted = chronological (sz0,sz1,...)
    if len(uniq)<3: return None
    sens_seeds=[]
    for seed in range(10):
        rng=np.random.RandomState(seed)
        # the anomaly model is calibrated identically for both splits
        idx=rng.permutation(len(Xi)); cut=len(Xi)//2
        Xi_cal=Xi[idx[:cut]]
        sc=StandardScaler().fit(Xi_cal)
        cov=EmpiricalCovariance().fit(sc.transform(Xi_cal))
        score=lambda A: cov.mahalanobis(sc.transform(A))
        tau=np.percentile(score(Xi_cal),95)   # FA target 0.05

        # the split only decides which seizures we test on, since tau comes from
        # interictal data either way.
        # random -> a random half; time -> the later half.
        k=len(uniq); nc=int(np.ceil(k/2))
        if split_type=='time':
            test_sz=uniq[nc:]                  # later seizures only
        else:
            perm=list(rng.permutation(uniq)); test_sz=perm[nc:]  # random half
        caught=[int(score(Xp[sid_p==z]).max()>=tau) for z in test_sz]
        if caught: sens_seeds.append(np.mean(caught))
    return float(np.mean(sens_seeds)) if sens_seeds else None

rand_s, time_s, pats=[],[],[]
for p in cohort:
    r=sens_under_split(p,'random'); t=sens_under_split(p,'time')
    if r is not None and t is not None:
        rand_s.append(r); time_s.append(t); pats.append(p)

rand_s,time_s=np.array(rand_s),np.array(time_s)
# heterogeneity costs us (1 - random); drift costs us (random - time)
het_gap=1-rand_s
temp_gap=rand_s-time_s
print(f"Patients: {len(pats)}")
print(f"Random-split sensitivity (generic heterogeneity):  {rand_s.mean():.3f}")
print(f"Time-split sensitivity   (heterogeneity + drift):  {time_s.mean():.3f}")
print(f"\nMean heterogeneity gap (1 - random):     {het_gap.mean():.3f}")
print(f"Mean temporal-drift gap (random - time): {temp_gap.mean():.3f}")
print(f"\n--> Of total failure, heterogeneity accounts for {het_gap.mean()/(het_gap.mean()+max(temp_gap.mean(),1e-9))*100:.0f}%, "
      f"temporal drift {max(temp_gap.mean(),0)/(het_gap.mean()+max(temp_gap.mean(),1e-9))*100:.0f}%")

json.dump({'patients':pats,'random':rand_s.tolist(),'time':time_s.tolist()},
          open(f'{OUT}/decomposition.json','w'),indent=2)

# ---- Figure 5: the two splits, patient by patient ----
fig,ax=plt.subplots(figsize=(6,4.2))
order=np.argsort(rand_s)
x=np.arange(len(pats))
ax.plot(x,rand_s[order],'o-',color='#2c3e50',label='random split (heterogeneity only)')
ax.plot(x,time_s[order],'s--',color='#c0392b',label='time-ordered split (+ temporal drift)')
ax.set_xticks(x); ax.set_xticklabels([pats[i] for i in order],rotation=90,fontsize=7)
ax.set_ylabel('Sensitivity'); ax.set_title('Decomposing non-exchangeability: heterogeneity vs temporal drift')
ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f'{OUT}/figures/fig5_decomposition.png',dpi=300,bbox_inches='tight')
plt.savefig(f'{OUT}/figures/fig5_decomposition.pdf',bbox_inches='tight')
import shutil; shutil.make_archive(f'{OUT}/all_figures','zip',f'{OUT}/figures')
print("\nSaved fig5_decomposition (PNG+PDF), updated all_figures.zip")

Patients: 18
Random-split sensitivity (generic heterogeneity):  0.946
Time-split sensitivity   (heterogeneity + drift):  0.963

Mean heterogeneity gap (1 - random):     0.054
Mean temporal-drift gap (random - time): -0.017

--> Of total failure, heterogeneity accounts for 100%, temporal drift 0%

Saved fig5_decomposition (PNG+PDF), updated all_figures.zip
